# Locations table

The purpose of this notebook is to create a table where have all possible locations of the students who registered to EIE or NMT during 2016-2025 and the locations of the test centers. The information about location contains:
- the old KOATUU code used until 2021, 
- the new KATOTTG code used from 2021, 
- the category of the administrative-territorial unit, 
- the English name of the corresponding region.

The original location data is highly inconsistent year over year, owing to the significant decommunization and decentralization that took place in Ukraine between 2016-2025. The decommunization process included removing Soviet-era location names and replacing them with names that reflect Ukraine’s national heritage. For instance, in 2016, a total of twenty-two villages bearing the name ‘с.Комсомольське’, which is derived from the Communist Party,underwent the renaming process. Also, in the open data location names are sometimes written in Russian instead of in Ukrainian (for example: ‘с.Фастовци’ is in Russian, while ‘с.Фастiвцi’ is the Ukrainian pronunciation)

## Loading libraries

In [1]:
import numpy as np
import pandas as pd
import importlib

import src.renaming_dictionaries as renam

## Collecting all possible locations

In [2]:
# load datasets
datasets = {}
years = range(2016, 2022)
for year in years:
    print(f"EIE {year} Loading...")
    if int(year) >= 2019:
        file_name = f'Odata{year}File.csv'
    else:
        file_name = f'OpenData{year}.csv'
    try:           
        dataset = pd.read_csv(f"../../data_loader/{year}/{file_name}", sep=";", encoding='utf-8', dtype = str)
    except:
        dataset = pd.read_csv(f"../../data_loader/{year}/{file_name}", sep=";", encoding='Windows 1251', dtype = str)
    datasets.update({year:dataset})
    print("success")

EIE 2016 Loading...
success
EIE 2017 Loading...
success
EIE 2018 Loading...
success
EIE 2019 Loading...
success
EIE 2020 Loading...
success
EIE 2021 Loading...
success


In [3]:
for year, dataset in datasets.items():
    #lowercase all columns and add year as attribute
    dataset.columns = [col.lower() for col in dataset.columns]
    #add year before merging datasets
    dataset['year'] = year

In [4]:
for year, dataset in datasets.items():
    print(year, dataset.shape)

2016 (268003, 107)
2017 (240889, 120)
2018 (335687, 126)
2019 (353813, 127)
2020 (379299, 127)
2021 (389323, 148)


In [5]:
for year, dataset in datasets.items():
    print([col for col in dataset.columns if col.endswith('regname')])

['regname', 'eoregname', 'ukrptregname', 'histptregname', 'mathptregname', 'physptregname', 'chemptregname', 'bioptregname', 'geoptregname', 'engptregname', 'frptregname', 'deuptregname', 'spptregname', 'rusptregname']
['regname', 'eoregname', 'ukrptregname', 'histptregname', 'mathptregname', 'physptregname', 'chemptregname', 'bioptregname', 'geoptregname', 'engptregname', 'fraptregname', 'deuptregname', 'spaptregname', 'rusptregname']
['regname', 'eoregname', 'ukrptregname', 'histptregname', 'mathptregname', 'physptregname', 'chemptregname', 'bioptregname', 'geoptregname', 'engptregname', 'fraptregname', 'deuptregname', 'spaptregname']
['regname', 'eoregname', 'ukrptregname', 'histptregname', 'mathptregname', 'physptregname', 'chemptregname', 'bioptregname', 'geoptregname', 'engptregname', 'fraptregname', 'deuptregname', 'spaptregname']
['regname', 'eoregname', 'ukrptregname', 'histptregname', 'mathptregname', 'physptregname', 'chemptregname', 'bioptregname', 'geoptregname', 'engptreg

In [6]:
types = ['', 'eo', 'ukrpt', 'umlpt', 'mathpt', 'histpt', 'physpt', 'mathstptregname', 'chempt', 'biopt', 'geopt', 'engpt', 'frapt', 'frpt', 'deupt', 'spapt', 'sppt', 'ruspt']

In [7]:
locations = pd.DataFrame(columns=['regname', 'areaname', 'tername'])

In [8]:
dataframes_to_concat = []

for year, dataset in datasets.items():
    for typ in types:
        new_col = {typ+'regname': 'regname', typ+'areaname':'areaname', typ+'tername': 'tername'}
        if all(column in dataset.columns for column in new_col):
            temp_data = dataset.loc[:, new_col.keys()].copy()
            temp_data.rename(columns=new_col, inplace=True)
            # Drop rows where all columns are NaN before adding to list
            temp_data = temp_data.dropna(how='all')
            dataframes_to_concat.append(temp_data)

if dataframes_to_concat:
    locations = pd.concat(dataframes_to_concat, ignore_index=True)
    locations.drop_duplicates(inplace=True)
    locations.reset_index(drop=True, inplace=True)
else:
    locations = pd.DataFrame(columns=['regname', 'areaname', 'tername'])

locations

,regname,areaname,tername
0,Запорізька область,Мелітопольський район,с.Терпіння
1,Хмельницька область,Красилівський район,м.Красилів
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка
3,Чернівецька область,м.Чернівці,Шевченківський район міста
4,Миколаївська область,Врадіївський район,с.Кумарі
...,...,...,...
22587,Житомирська область,Любарський район,с.Великий Браталів
22588,Хмельницька область,Ізяславський район,с.Влашанівка
22589,Львівська область,Яворівський район,с.Коханівка
22590,Житомирська область,Ємільчинський район,с.Бастова Рудня


# Codifier brush-up

In [9]:
code2020 = pd.read_excel('./location_data/KOATUU_26112020.xls',  dtype=str)
code2020

,TE,NP,NU
0,0100000000,NaN,АВТОНОМНА РЕСПУБЛІКА КРИМ/М.СІМФЕРОПОЛЬ
1,0110000000,NaN,МІСТА АВТОНОМНОЇ РЕСПУБЛІКИ КРИМ
2,0110100000,NaN,СІМФЕРОПОЛЬ
3,0110130000,NaN,РАЙОНИ М.СІМФЕРОПОЛЯ
4,0110136300,Р,ЗАЛІЗНИЧНИЙ
...,...,...,...
39415,8536990504,С,КАМИШЛИ
39416,8536990505,С,ПИРОГОВКА
39417,8536990507,С,ПОВОРОТНЕ
39418,8536990509,С,ФРОНТОВЕ


In [10]:
code2020['region_KOATUU'] = code2020['TE'].str[:2]
code2020['area_KOATUU'] = code2020['TE'].str[2:5]
code2020['local_KOATUU'] = code2020['TE'].str[5:]
code2020


,TE,NP,NU,region_KOATUU,area_KOATUU,local_KOATUU
0,0100000000,NaN,АВТОНОМНА РЕСПУБЛІКА КРИМ/М.СІМФЕРОПОЛЬ,01,000,00000
1,0110000000,NaN,МІСТА АВТОНОМНОЇ РЕСПУБЛІКИ КРИМ,01,100,00000
2,0110100000,NaN,СІМФЕРОПОЛЬ,01,101,00000
3,0110130000,NaN,РАЙОНИ М.СІМФЕРОПОЛЯ,01,101,30000
4,0110136300,Р,ЗАЛІЗНИЧНИЙ,01,101,36300
...,...,...,...,...,...,...
39415,8536990504,С,КАМИШЛИ,85,369,90504
39416,8536990505,С,ПИРОГОВКА,85,369,90505
39417,8536990507,С,ПОВОРОТНЕ,85,369,90507
39418,8536990509,С,ФРОНТОВЕ,85,369,90509


In [11]:
code2020['name_KOATUU'] = code2020['NU'].str.split('/').str[0].str.lower().str.capitalize()
code2020[code2020.area_KOATUU == '000'][['NU', 'region_KOATUU', 'name_KOATUU']]

,NU,region_KOATUU,name_KOATUU
0,АВТОНОМНА РЕСПУБЛІКА КРИМ/М.СІМФЕРОПОЛЬ,01,Автономна республіка крим
1328,ВІННИЦЬКА ОБЛАСТЬ/М.ВІННИЦЯ,05,Вінницька область
3508,ВОЛИНСЬКА ОБЛАСТЬ/М.ЛУЦЬК,07,Волинська область
4870,ДНІПРОПЕТРОВСЬКА ОБЛАСТЬ/М.ДНІПРО,12,Дніпропетровська область
6645,ДОНЕЦЬКА ОБЛАСТЬ/М.ДОНЕЦЬК,14,Донецька область
8272,ЖИТОМИРСЬКА ОБЛАСТЬ/М.ЖИТОМИР,18,Житомирська область
10367,ЗАКАРПАТСЬКА ОБЛАСТЬ/М.УЖГОРОД,21,Закарпатська область
11303,ЗАПОРІЗЬКА ОБЛАСТЬ/М.ЗАПОРІЖЖЯ,23,Запорізька область
12483,ІВАНО-ФРАНКІВСЬКА ОБЛАСТЬ/М.ІВАНО-ФРАНКІВСЬК,26,Івано-франківська область
13716,КИЇВСЬКА ОБЛАСТЬ/М.КИЇВ,32,Київська область


In [12]:
code2020 = code2020[~code2020['region_KOATUU'].isin(['01', '85'])]

In [13]:
code2020 = code2020[~code2020['name_KOATUU'].str.lower().str.startswith('міста ')]
code2020 = code2020[~code2020['name_KOATUU'].str.lower().str.startswith('райони ')]
code2020 = code2020[~code2020['name_KOATUU'].str.lower().str.startswith('селища ')]
code2020 = code2020[~code2020['name_KOATUU'].str.lower().str.startswith('населенi ')]
code2020 = code2020[~code2020['name_KOATUU'].str.lower().str.startswith('селища, ')]
code2020 = code2020[~code2020['name_KOATUU'].str.lower().str.startswith('сільради ')]
code2020 = code2020.reset_index(drop=True)

In [14]:
code2020.NP.unique()

array([nan, 'М', 'Т', 'С', 'Щ', 'Р', 'C'], dtype=object)

In [15]:
code2020.NP = code2020.NP.str.replace('C', 'С')

In [16]:
code2020.loc[code2020['NP'] == 'М', 'name_KOATUU'] = 'м.' + code2020.loc[code2020['NP'] == 'М', 'name_KOATUU']
code2020.loc[code2020['NP'] == 'С', 'name_KOATUU'] = 'с.' + code2020.loc[code2020['NP'] == 'С', 'name_KOATUU']
code2020.loc[code2020['NP'] == 'Щ', 'name_KOATUU'] = 'с-ще ' + code2020.loc[code2020['NP'] == 'Щ', 'name_KOATUU']
code2020.loc[code2020['NP'] == 'Т', 'name_KOATUU'] = 'смт ' + code2020.loc[code2020['NP'] == 'Т', 'name_KOATUU']

In [17]:
set(code2020[code2020.area_KOATUU == '000'].name_KOATUU.str.lower())^set(locations.regname.str.lower())

set()

In [18]:
locations_region = locations.merge(code2020[['region_KOATUU']], left_on=locations['regname'].str.lower(), right_on=code2020['name_KOATUU'].str.lower(), how='left')
locations_region = locations_region.drop('key_0', axis=1)
locations_region.head()

,regname,areaname,tername,region_KOATUU
0,Запорізька область,Мелітопольський район,с.Терпіння,23
1,Хмельницька область,Красилівський район,м.Красилів,68
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73
4,Миколаївська область,Врадіївський район,с.Кумарі,48


In [19]:
locations_region[['regname', 'region_KOATUU']].value_counts()

regname                    region_KOATUU
Львівська область          46               1787
Вінницька область          05               1338
Житомирська область        18               1274
Полтавська область         53               1215
Хмельницька область        68               1207
Харківська область         63               1176
Дніпропетровська область   12               1120
Рівненська область         56               1102
Волинська область          07               1079
Тернопільська область      61               1039
Київська область           32               1027
Одеська область            51                935
Чернігівська область       74                884
Івано-Франківська область  26                823
Запорізька область         23                800
Донецька область           14                754
Сумська область            59                750
Черкаська область          71                734
Закарпатська область       21                689
Кіровоградська область     3

# Matching areas

In [20]:
locations_region['areaname_new'] = locations_region['areaname']
locations_region.head(25)

,regname,areaname,tername,region_KOATUU,areaname_new
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район
5,Донецька область,Донецька область,м.Дружківка,14,Донецька область
6,Тернопільська область,Тернопільська область,м.Тернопіль,61,Тернопільська область
7,Дніпропетровська область,Дніпропетровська область,м.Нікополь,12,Дніпропетровська область
8,Кіровоградська область,Новгородківський район,с.Тарасівка,35,Новгородківський район
9,Дніпропетровська область,м.Дніпропетровськ,Жовтневий район міста,12,м.Дніпропетровськ


Sometimes they wrongly put the name of regions in the second column. We fix it here

In [21]:
locations_region['areaname_new'] = locations_region.apply(lambda row: row['tername'] if ('область' in row['areaname_new'] or 'м.Київ' in row['areaname_new']) else row['areaname_new'], axis=1)

In [22]:
locations_region['areaname_new'] = locations_region.apply(lambda row: row['areaname_new'].split('. ')[0] if ('. ' in row['areaname_new']) else row['areaname_new'], axis=1)

In [23]:
dct = {}
for code in code2020.region_KOATUU.unique():
    troubles = set(locations_region[(locations_region.region_KOATUU == code)].areaname_new.str.lower())-set(code2020[(code2020.region_KOATUU == code)&(code2020.area_KOATUU != '000')&(code2020.local_KOATUU == '00000')].name_KOATUU.str.lower())
    if troubles:
        dct[code] = troubles
print(dct)


{'12': {'м.кривий ріг', 'дніпропетровський район', "м.кам'янське", 'м.орджонікідзе', 'м.дніпро', 'м.дніпродзержинськ', 'м.дніпропетровськ'}, '14': {'м.маріуполь', 'м.дзержинськ', 'м.димитров', 'володарський район', 'м.артемівськ', 'красноармійський район', 'м.красний лиман', 'артемівський район', 'м.красноармійськ', 'першотравневий район'}, '18': {'м.житомир', 'червоноармійський район', 'володарськ-волинський район'}, '21': {'м.мукачеве'}, '23': {'м.запоріжжя', 'куйбишевський район'}, '32': {'м.переяслав-хмельницький'}, '35': {'м.кропивницький', 'кіровоградський район', 'м.кіровоград', 'ульяновський район'}, '46': {'м.львів'}, '48': {'м.миколаїв', 'жовтневий район'}, '51': {'красноокнянський район', 'м.одеса', 'м.іллічівськ', 'фрунзівський район', 'котовський район', 'комінтернівський район', 'м.котовськ'}, '53': {'м.кременчук', 'м.полтава', 'м.комсомольськ'}, '56': {'м.кузнецовськ'}, '59': {'м.суми'}, '63': {'м.харків'}, '65': {'м.херсон', 'цюрупинський район'}, '71': {'м.черкаси'}, '

In [24]:
for code in renam.dct_rename_area:
    for name in renam.dct_rename_area[code]:
        locations_region.loc[(locations_region.region_KOATUU == code) & (locations_region['areaname_new'] == name), 'areaname_new'] = renam.dct_rename_area[code][name]

In [25]:
for city in renam.dct_cities:
    code2020.loc[(code2020['local_KOATUU'] == '00000') & (code2020['name_KOATUU'].str.lower() == city), 'name_KOATUU'] = renam.dct_cities[city]


In [26]:
for ray in renam.kyiv_dict:
    locations_region.loc[(locations_region.regname == 'м.Київ') & (locations_region.areaname_new.str.lower() == ray), 'areaname_new'] = renam.kyiv_dict[ray]


In [27]:
dct = {}
for code in code2020.region_KOATUU.unique():
    troubles = set(locations_region[(locations_region.region_KOATUU == code)].areaname_new.str.lower())-set(code2020[(code2020.region_KOATUU == code)&(code2020.area_KOATUU != '000')&(code2020.local_KOATUU == '00000')].name_KOATUU.str.lower())
    if troubles:
        dct[code] = troubles
print(dct)

{}


In [28]:
locations_area = locations_region.merge(code2020[['area_KOATUU']], left_on=[locations_region['region_KOATUU'], locations_region['areaname_new'].str.lower()], right_on=[code2020['region_KOATUU'], code2020['name_KOATUU'].str.lower()], how='left')
locations_area = locations_area.drop(columns = ['key_0', 'key_1'], axis=1)
locations_area.head()

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223


In [29]:
locations_area[(locations_area.area_KOATUU.isna())]

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU


# Matching tername

In [30]:
locations_area['tername_new'] = locations_area['tername']
locations_area['tername_new'] = locations_area['tername_new'].str.replace(r'\(.*\)', '', regex = True).str.strip()
locations_area['tername_new'] = locations_area['tername_new'].str.replace(' район міста','')
locations_area['tername_new']

0                с.Терпіння
1                м.Красилів
2               с.Дмитрівка
3            Шевченківський
4                  с.Кумарі
                ...        
22587    с.Великий Браталів
22588          с.Влашанівка
22589           с.Коханівка
22590       с.Бастова Рудня
22591            м.Чернівці
Name: tername_new, Length: 22592, dtype: object

In [31]:
locations_area['code'] = locations_area['region_KOATUU']+locations_area['area_KOATUU']
locations_area

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230,с.Терпіння,23230
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227,м.Красилів,68227
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238,с.Дмитрівка,12238
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101,Шевченківський,73101
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223,с.Кумарі,48223
...,...,...,...,...,...,...,...,...
22587,Житомирська область,Любарський район,с.Великий Браталів,18,Любарський район,231,с.Великий Браталів,18231
22588,Хмельницька область,Ізяславський район,с.Влашанівка,68,Ізяславський район,221,с.Влашанівка,68221
22589,Львівська область,Яворівський район,с.Коханівка,46,Яворівський район,258,с.Коханівка,46258
22590,Житомирська область,Ємільчинський район,с.Бастова Рудня,18,Ємільчинський район,217,с.Бастова Рудня,18217


In [32]:
code2020['code'] = code2020['region_KOATUU']+code2020['area_KOATUU']
code2020

,TE,NP,NU,region_KOATUU,area_KOATUU,local_KOATUU,name_KOATUU,code
0,0500000000,NaN,ВІННИЦЬКА ОБЛАСТЬ/М.ВІННИЦЯ,05,000,00000,Вінницька область,05000
1,0510100000,М,ВІННИЦЯ,05,101,00000,м.Вінниця,05101
2,0510141000,Т,ДЕСНА,05,101,41000,смт Десна,05101
3,0510300000,М,ЖМЕРИНКА,05,103,00000,м.Жмеринка,05103
4,0510390001,С,ЖУКІВЦІ,05,103,90001,с.Жуківці,05103
...,...,...,...,...,...,...,...,...
36847,8038200000,Р,ПЕЧЕРСЬКИЙ,80,382,00000,Печерський,80382
36848,8038500000,Р,ПОДІЛЬСЬКИЙ,80,385,00000,Подільський,80385
36849,8038600000,Р,СВЯТОШИНСЬКИЙ,80,386,00000,Святошинський,80386
36850,8038900000,Р,СОЛОМ'ЯНСЬКИЙ,80,389,00000,Солом'янський,80389


In [33]:
renaming = pd.read_csv('./location_data/Renaming.csv')
renaming

,New,Old,areaname,regname,year
0,с.Азовське,с.Луначарське,Бердянський район,Запорізька область,2016
1,с.Армашівка,с.Орджонікідзе,Ширяївський район,Одеська область,2016
2,с.Високе,с.Артемівка,Кегичівський район,Харківська область,2016
3,с.Вишневе,с.Жовтневе,Калинівський район,Вінницька область,2016
4,с.Єрмішкове,с.Дзержинське,Великомихайлівський район,Одеська область,2016
...,...,...,...,...,...
1536,м.Яремче,м.Яремча,м.Яремче,Івано-Франківська область,2006
1537,с.Яромель,с.Йосипівка,Ківерцівський район,Волинська область,1989
1538,с.Ясинівка,с.Ясенівка,Дубенський район,Рівненська область,2013
1539,с.Ятвяги,с.Прибілля,Жидачівський район,Львівська область,2015


In [34]:
set(code2020[code2020.area_KOATUU == '000'].name_KOATUU.str.lower())^set(renaming.regname.str.lower())

{'м.київ'}

In [35]:
renam_code = renaming.merge(code2020[['region_KOATUU']], left_on=renaming['regname'].str.lower(), right_on=code2020['name_KOATUU'].str.lower(), how='left')
renam_code = renam_code.drop('key_0', axis=1)
renam_code

,New,Old,areaname,regname,year,region_KOATUU
0,с.Азовське,с.Луначарське,Бердянський район,Запорізька область,2016,23
1,с.Армашівка,с.Орджонікідзе,Ширяївський район,Одеська область,2016,51
2,с.Високе,с.Артемівка,Кегичівський район,Харківська область,2016,63
3,с.Вишневе,с.Жовтневе,Калинівський район,Вінницька область,2016,05
4,с.Єрмішкове,с.Дзержинське,Великомихайлівський район,Одеська область,2016,51
...,...,...,...,...,...,...
1536,м.Яремче,м.Яремча,м.Яремче,Івано-Франківська область,2006,26
1537,с.Яромель,с.Йосипівка,Ківерцівський район,Волинська область,1989,07
1538,с.Ясинівка,с.Ясенівка,Дубенський район,Рівненська область,2013,56
1539,с.Ятвяги,с.Прибілля,Жидачівський район,Львівська область,2015,46


In [36]:
dct = {}
for code in code2020.region_KOATUU.unique():
    troubles = set(renam_code[(renam_code.region_KOATUU == code)].areaname.str.lower())-set(code2020[(code2020.region_KOATUU == code)&(code2020.area_KOATUU != '000')&(code2020.local_KOATUU == '00000')].name_KOATUU.str.lower())
    if troubles:
        dct[code] = troubles
print(dct)

{}


In [37]:
renam_code = renam_code.merge(code2020[['area_KOATUU']], left_on=[renam_code['region_KOATUU'], renam_code['areaname'].str.lower()], right_on=[code2020['region_KOATUU'], code2020['name_KOATUU'].str.lower()], how='left')
renam_code = renam_code.drop(columns = ['key_0', 'key_1'], axis=1)
renam_code

,New,Old,areaname,regname,year,region_KOATUU,area_KOATUU
0,с.Азовське,с.Луначарське,Бердянський район,Запорізька область,2016,23,206
1,с.Армашівка,с.Орджонікідзе,Ширяївський район,Одеська область,2016,51,254
2,с.Високе,с.Артемівка,Кегичівський район,Харківська область,2016,63,231
3,с.Вишневе,с.Жовтневе,Калинівський район,Вінницька область,2016,05,216
4,с.Єрмішкове,с.Дзержинське,Великомихайлівський район,Одеська область,2016,51,216
...,...,...,...,...,...,...,...
1536,м.Яремче,м.Яремча,м.Яремче,Івано-Франківська область,2006,26,110
1537,с.Яромель,с.Йосипівка,Ківерцівський район,Волинська область,1989,07,218
1538,с.Ясинівка,с.Ясенівка,Дубенський район,Рівненська область,2013,56,216
1539,с.Ятвяги,с.Прибілля,Жидачівський район,Львівська область,2015,46,215


In [38]:
renam_code['code'] = renam_code['region_KOATUU']+renam_code['area_KOATUU']
renam_code

,New,Old,areaname,regname,year,region_KOATUU,area_KOATUU,code
0,с.Азовське,с.Луначарське,Бердянський район,Запорізька область,2016,23,206,23206
1,с.Армашівка,с.Орджонікідзе,Ширяївський район,Одеська область,2016,51,254,51254
2,с.Високе,с.Артемівка,Кегичівський район,Харківська область,2016,63,231,63231
3,с.Вишневе,с.Жовтневе,Калинівський район,Вінницька область,2016,05,216,05216
4,с.Єрмішкове,с.Дзержинське,Великомихайлівський район,Одеська область,2016,51,216,51216
...,...,...,...,...,...,...,...,...
1536,м.Яремче,м.Яремча,м.Яремче,Івано-Франківська область,2006,26,110,26110
1537,с.Яромель,с.Йосипівка,Ківерцівський район,Волинська область,1989,07,218,07218
1538,с.Ясинівка,с.Ясенівка,Дубенський район,Рівненська область,2013,56,216,56216
1539,с.Ятвяги,с.Прибілля,Жидачівський район,Львівська область,2015,46,215,46215


In [39]:
renaming_dict = {(row['Old'], row['code']): row['New'] for _, row in renam_code.iterrows()}
locations_area['tername_new'] = locations_area.apply(lambda row: renaming_dict.get((row['tername_new'], row['code']), row['tername_new']), axis=1)

In [40]:
for cod in locations_area.code.unique():
    troubles = set(locations_area[(locations_area.code == cod)].tername_new.str.lower())-set(code2020[code2020.code == cod].name_KOATUU.str.lower())
    if troubles:
        print(f'{repr(cod)}:{troubles},')

'68227':{'с.дружнє'},
'73101':{'першотравневий', 'шевченківський', 'садгірський'},
'14117':{'смт олексіїво-дружківка'},
'12101':{'ленінський', 'красногвардійський', 'бабушкінський', 'жовтневий', 'кіровський'},
'46258':{'с.щеплати', 'с.руда краковецька'},
'26240':{'с.саджавка'},
'68239':{'с.старий кривин', "с.д'яків", 'с.голики'},
'51223':{'с.миколаівка'},
'63216':{'с-ще вірівка', 'с-ще новоолександрівка', 'с-ще профінтерн'},
'14209':{'с.зайцеве', 'с.переізне', 'с-ще опитне', 'с-ще зеленопілля', 'с.іванівське', 'с.відродження', 'с.покровське', 'с-ще хромове', 'с.іванград', 'с.кліщіївка', 'с-ще володимирівка', 'с.клинове'},
'46212':{'с.отриничі'},
'71234':{'с.летицівка'},
'63101':{'червонозаводський', 'дзержинський', 'комінтернівський', 'ленінський', 'орджонікідзевський', 'фрунзенський', 'жовтневий'},
'35222':{'с.золоми'},
'05101':{'староміський', 'ленінський', 'замостянський'},
'74241':{'с.билорічиця'},
'14141':{'м.миколаївка'},
'56212':{'с.угольці'},
'53232':{'с.декабристів'},
'65101':

In [41]:
for code in renam.dct_replace:
    for place in renam.dct_replace[code]:
        locations_area.loc[(locations_area.code == code) & (locations_area['tername_new'] == place), 'tername_new'] = renam.dct_replace[code][place]

In [42]:
def change_area_KOATUU(old_code, place, code, dataset):
    dataset.loc[(dataset.code == old_code)&(dataset['tername_new'] == place), 'area_KOATUU'] = code[2:]
    dataset.loc[(dataset.code == old_code)&(dataset['tername_new'] == place), 'code'] = code
    values = code2020.loc[(code2020.code == code)&(code2020.local_KOATUU == '00000'), 'name_KOATUU'].unique()
    if len(values) == 1:
        dataset.loc[(dataset.code == code)&(dataset['tername_new'] == place), 'areaname_new'] = values[0]
    else:
        raise Exception(f'More than one value:{old_code}, {place}, {code}, {values}')

In [43]:
for item in renam.place_change:
    change_area_KOATUU(item[0], item[1], item[2], locations_area)

In [44]:
for code in renam.dct_replace:
    for place in renam.dct_replace[code]:
        locations_area.loc[(locations_area.code == code) & (locations_area['tername_new'] == place), 'tername_new'] = renam.dct_replace[code][place]

In [45]:
locations_KOATUU= locations_area.merge(code2020[['local_KOATUU']], left_on=[locations_area['region_KOATUU'], locations_area['area_KOATUU'], locations_area['tername_new'].str.lower()], right_on=[code2020['region_KOATUU'], code2020['area_KOATUU'], code2020['name_KOATUU'].str.lower()], how='left')
locations_KOATUU = locations_KOATUU.drop(columns = ['key_0', 'key_1', 'key_2'], axis=1)
locations_KOATUU.head()

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230,с.Терпіння,23230,85101
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227,м.Красилів,68227,10100
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238,с.Дмитрівка,12238,81501
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101,м.Чернівці,73101,00000
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223,с.Кумарі,48223,83001


In [46]:
locations_KOATUU['KOATUU'] = locations_KOATUU['region_KOATUU']+locations_KOATUU['area_KOATUU']+locations_KOATUU['local_KOATUU']
locations_KOATUU.head()

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230,с.Терпіння,23230,85101,2323085101
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227,м.Красилів,68227,10100,6822710100
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238,с.Дмитрівка,12238,81501,1223881501
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101,м.Чернівці,73101,00000,7310100000
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223,с.Кумарі,48223,83001,4822383001


# Match with the new system of code2020

In [47]:
comparison = pd.read_excel('./location_data/Comparison table.xlsx',  dtype=str, skiprows=2)
comparison

,Кодифікатор,Код об'єкта КОАТУУ,Категорія об’єкта,Назва об’єкта
0,UA01000000000013043,0100000000,О,Автономна Республіка Крим
1,UA01020000000022387,0120400000,Р,Бахчисарайський
2,UA01020010000048857,8536990200,Н,Андріївська
3,UA01020010010075540,8536990201,С,Андріївка
4,UA01020010020030666,8536990203,Х,Сонячний
...,...,...,...,...
31753,UA85000000000065278,8500000000,К,Севастополь
31754,UA85000000000155841,8536300000,В,Балаклавський
31755,UA85000000000259923,8536400000,В,Гагарінський
31756,UA85000000000334608,8536600000,В,Ленінський


In [48]:
comparison.rename(columns = {'Кодифікатор':'KATOTTG',"Код об'єкта КОАТУУ ":'KOATUU','Категорія об’єкта':'category', 'Назва об’єкта':'name'}, inplace=True)
comparison

,KATOTTG,KOATUU,category,name
0,UA01000000000013043,0100000000,О,Автономна Республіка Крим
1,UA01020000000022387,0120400000,Р,Бахчисарайський
2,UA01020010000048857,8536990200,Н,Андріївська
3,UA01020010010075540,8536990201,С,Андріївка
4,UA01020010020030666,8536990203,Х,Сонячний
...,...,...,...,...
31753,UA85000000000065278,8500000000,К,Севастополь
31754,UA85000000000155841,8536300000,В,Балаклавський
31755,UA85000000000259923,8536400000,В,Гагарінський
31756,UA85000000000334608,8536600000,В,Ленінський


In [49]:
comparison_loc = comparison[comparison.category.isin(['С', 'М', 'В', 'Т', 'Х'])]

In [50]:
locations_KOATUU.loc[(locations_KOATUU.KOATUU == '5120280501'), 'local_KOATUU'] = '80401'
locations_KOATUU.loc[(locations_KOATUU.KOATUU == '5120280501'), 'KOATUU'] = '5120280401'
    

In [51]:
# locations_KATOTTG = locations_KOATUU.merge(comparison_loc[['KATOTTG']], left_on=[locations_KOATUU['KOATUU_2020']], right_on=[comparison_loc['KOATUU_2020']], how='left')
locations_KATOTTG = locations_KOATUU.merge(comparison_loc[['KATOTTG', 'category', 'KOATUU']], on=['KOATUU'], how='left')
# locations_KATOTTG  = locations_KATOTTG.drop(columns = ['key_0'], axis=1)
locations_KATOTTG.head()

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230,с.Терпіння,23230,85101,2323085101,UA23080270010078454,С
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227,м.Красилів,68227,10100,6822710100,UA68040210010032567,М
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238,с.Дмитрівка,12238,81501,1223881501,UA12140170040016918,С
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101,м.Чернівці,73101,00000,7310100000,UA73060610010033137,М
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223,с.Кумарі,48223,83001,4822383001,UA48080050190079797,С


In [52]:
locations_KATOTTG.category.unique()

array(['С', 'М', 'В', 'Т', 'Х', nan], dtype=object)

check na

In [53]:
locations_KATOTTG[(locations_KATOTTG.KATOTTG.isna())]

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category
13686,Дніпропетровська область,Дніпропетровська область,Петропавлівський район,12,Петропавлівський район,238,Петропавлівський район,12238,00000,1223800000,NaN,NaN
16272,Львівська область,Львівська область,Сокальський район,46,Сокальський район,248,Сокальський район,46248,00000,4624800000,NaN,NaN


these two tuples are not valid, since the final location couldn't be area. So we drop them

In [54]:
locations_KATOTTG= locations_KATOTTG[(locations_KATOTTG.KATOTTG.notna())]
locations_KATOTTG.reset_index(drop=True, inplace=True)
locations_KATOTTG

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230,с.Терпіння,23230,85101,2323085101,UA23080270010078454,С
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227,м.Красилів,68227,10100,6822710100,UA68040210010032567,М
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238,с.Дмитрівка,12238,81501,1223881501,UA12140170040016918,С
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101,м.Чернівці,73101,00000,7310100000,UA73060610010033137,М
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223,с.Кумарі,48223,83001,4822383001,UA48080050190079797,С
...,...,...,...,...,...,...,...,...,...,...,...,...
22692,Житомирська область,Любарський район,с.Великий Браталів,18,Любарський район,231,с.Великий Браталів,18231,55106,1823155106,UA18040290070047586,С
22693,Хмельницька область,Ізяславський район,с.Влашанівка,68,Ізяславський район,221,с.Влашанівка,68221,86602,6822186602,UA68060250040099385,С
22694,Львівська область,Яворівський район,с.Коханівка,46,Яворівський район,258,с.Коханівка,46258,87803,4625887803,UA46140110360079694,С
22695,Житомирська область,Ємільчинський район,с.Бастова Рудня,18,Ємільчинський район,217,с.Бастова Рудня,18217,80403,1821780403,UA18080030040080438,С


In [55]:
locations_KATOTTG.columns

Index(['regname', 'areaname', 'tername', 'region_KOATUU', 'areaname_new',
       'area_KOATUU', 'tername_new', 'code', 'local_KOATUU', 'KOATUU',
       'KATOTTG', 'category'],
      dtype='object')

# 2022

In [56]:
dataset_2022 = pd.read_csv(f"../../data_loader/2022/Odata2022File.csv", sep=";", encoding='utf-8')
dataset_2022

,OUTID,Birth,SexTypeName,RegName,AREANAME,TERNAME,RegTypeName,TerTypeName,EONAME,EOTypeName,...,Block1Ball,Block2,Block2Ball100,Block2Ball,Block3,Block3Ball100,Block3Ball,PTRegName,PTAreaName,PTTerName
0,d60381f3-8d71-441e-817e-49b9fa8b43dd,2005,чоловіча,Львівська область,Яворівський район,с.Гусаків,Випускник закладу загальної середньої освіти 2...,"селище, село","Гусаківський навчально-виховний комплекс ""Зага...",навчально-виховний комплекс,...,14.0,Історія України,"147,0",15.0,Математика,"128,0",5.0,Львівська область,"м.Львів, Залізничний район міста",Залізничний район міста
1,eb25a9fc-b757-4321-a2b4-ebb1b635397d,2005,чоловіча,Львівська область,Яворівський район,с.Гусаків,Випускник закладу загальної середньої освіти 2...,"селище, село","Гусаківський навчально-виховний комплекс ""Зага...",навчально-виховний комплекс,...,11.0,Історія України,"149,0",17.0,Математика,"136,0",8.0,Львівська область,"м.Львів, Залізничний район міста",Залізничний район міста
2,1cb161bd-51ed-4d24-b605-1d45db63cada,2005,жіноча,Львівська область,Яворівський район,с.Гусаків,Випускник закладу загальної середньої освіти 2...,"селище, село","Гусаківський навчально-виховний комплекс ""Зага...",навчально-виховний комплекс,...,25.0,Історія України,"158,0",24.0,Математика,"185,0",28.0,Львівська область,"м.Львів, Залізничний район міста",Залізничний район міста
3,0311b8d8-67bb-49a4-a0b9-f049b7ef4184,2005,жіноча,Львівська область,Яворівський район,с.Гусаків,Випускник закладу загальної середньої освіти 2...,"селище, село","Гусаківський навчально-виховний комплекс ""Зага...",навчально-виховний комплекс,...,20.0,Історія України,"146,0",14.0,Математика,"148,0",14.0,Львівська область,"м.Львів, Залізничний район міста",Залізничний район міста
4,a8b35a53-feac-4e42-aed8-8d6ffab7decf,2005,чоловіча,Львівська область,Яворівський район,с.Гусаків,Випускник закладу загальної середньої освіти 2...,"селище, село","Гусаківський навчально-виховний комплекс ""Зага...",навчально-виховний комплекс,...,24.0,Історія України,"144,0",13.0,Математика,"144,0",12.0,Львівська область,"м.Львів, Залізничний район міста",Залізничний район міста
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234099,6cbe3454-80f0-42b5-8d6d-c2ead687d804,2005,чоловіча,Волинська область,Володимирський район,с.Заболотці,Випускник закладу загальної середньої освіти 2...,"селище, село",Заболотцівський ліцей Литовезької сільської ра...,середня загальноосвітня школа,...,13.0,Історія України,"136,0",9.0,Математика,"148,0",14.0,Волинська область,м.Нововолинськ,м.Нововолинськ
234100,b897557b-c01b-49d4-8624-b351aaafe5ff,2005,жіноча,Волинська область,Володимирський район,с.Заболотці,Випускник закладу загальної середньої освіти 2...,"селище, село",Заболотцівський ліцей Литовезької сільської ра...,середня загальноосвітня школа,...,31.0,Історія України,"170,0",29.0,Математика,"162,0",22.0,Волинська область,м.Нововолинськ,м.Нововолинськ
234101,dcec644b-dcec-47ee-9971-6beebd1a929b,2005,чоловіча,Волинська область,Володимирський район,с.Заболотці,Випускник закладу загальної середньої освіти 2...,"селище, село",Заболотцівський ліцей Литовезької сільської ра...,середня загальноосвітня школа,...,21.0,Історія України,"152,0",20.0,Математика,"154,0",19.0,Волинська область,м.Нововолинськ,м.Нововолинськ
234102,cb4156bb-a624-4274-9d38-28be3096f6b9,2004,жіноча,Волинська область,Володимирський район,с.Заболотці,Випускник закладу загальної середньої освіти 2...,"селище, село",Заболотцівський ліцей Литовезької сільської ра...,середня загальноосвітня школа,...,12.0,Історія України,"131,0",7.0,Математика,"131,0",6.0,Волинська область,м.Нововолинськ,м.Нововолинськ


In [57]:
dataset_2023 = pd.read_csv(f"../../data_loader//2023/Odata2023File.csv", sep=";", encoding='utf-8')
dataset_2023

,outid,Birth,SexTypeName,RegName,AreaName,TerName,RegTypeName,TerTypeName,EOName,EOTypeName,...,DeuBlockStatus,DeuBlockBall100,DeuBlockBall,SpaBlock,SpaBlockStatus,SpaBlockBall100,SpaBlockBall,PTRegName,PTAreaName,PTTerName
0,98e9d1c5-c3e3-41ed-b8ef-8e42f293b8e2,2006,жіноча,Запорізька область,м.Запоріжжя,Шевченківський район міста,Випускник закладу загальної середньої освіти 2...,місто,Запорізька гімназія №93 Запорізької міської ра...,гімназія,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Одеська область,"м.Одеса, Малиновський район міста",Малиновський район міста
1,98e9ff41-20ec-4cb0-823c-b44580578ec0,2006,жіноча,Херсонська область,м.Херсон,Корабельний район міста,Випускник закладу загальної середньої освіти 2...,місто,Херсонська гімназія №1 Херсонської міської ради,гімназія,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Данія,м.Орхус,м.Орхус
2,98e06b8f-a755-4101-9379-d0f9332f6a23,2006,жіноча,Дніпропетровська область,м.Дніпро,Центральний район міста,Випускник закладу загальної середньої освіти 2...,місто,"Комунальний заклад освіти ""Навчально-виховний ...",навчально-виховний комплекс,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Дніпропетровська область,"м.Дніпро, Соборний район міста",Соборний район міста
3,990e6cee-d60e-4d7f-a71f-6cba3101ed35,2003,жіноча,Запорізька область,м.Запоріжжя,Олександрівський район міста,Випускник минулих років,місто,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Запорізька область,"м.Запоріжжя, Олександрівський район міста",Олександрівський район міста
4,98e62f06-f6e4-4b8f-8086-10b55a9335b1,2006,чоловіча,м.Київ,м.Київ,Деснянський район міста,Випускник закладу загальної середньої освіти 2...,місто,"Навчально-виховний комплекс ""Дошкільний навчал...",навчально-виховний комплекс,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,м.Київ,"м.Київ, Голосіївський район міста",Голосіївський район міста
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288930,98e83726-5473-40e0-a7c3-30f29f2bb0ec,2006,чоловіча,Донецька область,Волноваський район,смт Велика Новосілка,Випускник закладу загальної середньої освіти 2...,селище міського типу,Великоновосілківський заклад загальної середнь...,середня загальноосвітня школа,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Тернопільська область,м.Тернопіль,м.Тернопіль
288931,98de6ea2-28f3-4951-990a-9a3ff45449de,2004,чоловіча,Волинська область,Ковельський район,м.Ковель,Випускник минулих років,місто,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Волинська область,м.Ковель,м.Ковель
288932,98e883cb-d2b7-4bc7-b86e-3ac16b36a0ff,2006,жіноча,Львівська область,м.Львів,Сихівський район міста,Випускник закладу загальної середньої освіти 2...,місто,Середня загальноосвітня школа №13,середня загальноосвітня школа,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Львівська область,"м.Львів, Сихівський район міста",Сихівський район міста
288933,9909aefa-b1e9-4eef-a911-25fae0b9f70d,2004,чоловіча,Харківська область,Лозівський район,м.Лозова,Випускник закладу загальної середньої освіти 2...,місто,Лозівський центр професійної освіти Харківсько...,заклад професійної (професійно-технічної) освіти,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Харківська область,м.Лозова,м.Лозова


In [58]:
dataset_2024 = pd.read_csv(f"../../data_loader/2024/Odata2024File.csv", sep=";", encoding='utf-8')
dataset_2024

,outid,Birth,SexTypeName,RegName,AreaName,TerName,RegTypeName,TerTypeName,EOName,EOTypeName,...,SpaBlockStatus,SpaBlockBall100,SpaBlockBall,UkrLitBlock,UkrLitBlockStatus,UkrLitBlockBall100,UkrLitBlockBall,PTRegName,PTAreaName,PTTerName
0,9b995d13-de3d-47c2-9b4e-004025346a49,2006,жіноча,Черкаська область,Черкаський район,с.Ліпляве,Випускник загальноосвітнього навчального закла...,"селище, село","Комунальний заклад ""Ліплявський ліцей"" Ліплявс...",ліцей,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Черкаська область,м.Канів,м.Канів
1,9c0a5f77-73e4-475d-aff9-006da514737b,2007,чоловіча,Дніпропетровська область,Нікопольський район,м.Нікополь,Випускник загальноосвітнього навчального закла...,місто,"Відокремлений структурний підрозділ ""Нікопольс...",заклад фахової передвищої освіти,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Дніпропетровська область,м.Покров,м.Покров
2,9bb9c262-2188-475c-949a-000b1a194cbb,2006,жіноча,Івано-Франківська область,Івано-Франківський район,м.Івано-Франківськ,Випускник загальноосвітнього навчального закла...,місто,"Фаховий коледж Закладу вищої освіти ""Університ...",заклад фахової передвищої освіти,...,NaN,NaN,NaN,Українська література,Зараховано,"125,0",12.0,Польща,м.Домброва Горнича,м.Домброва Горнича
3,9b982071-10d6-4caa-b80a-004675647a31,2007,жіноча,Дніпропетровська область,м.Дніпро,Чечелівський район міста,Випускник загальноосвітнього навчального закла...,місто,"Комунальний заклад освіти ""Середня загальноосв...",середня загальноосвітня школа,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Латвія,м.Рига,м.Рига
4,9b9fc04c-570f-4cb9-903c-0007200fe870,2007,жіноча,Львівська область,Самбірський район,с.Вовче,Випускник загальноосвітнього навчального закла...,"селище, село",ВОВЧЕНСЬКИЙ ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬОЇ ОСВІТИ ...,середня загальноосвітня школа,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Львівська область,Самбірський район,м.Старий Самбір
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
312503,9b9de379-0d9c-4208-9ff3-ff9c53ad2db4,2006,жіноча,Житомирська область,Бердичівський район,м.Бердичів,Випускник загальноосвітнього навчального закла...,місто,Ліцей № 4 м. Бердичева Житомирської області,ліцей,...,NaN,NaN,NaN,Українська література,Зараховано,"170,0",36.0,Житомирська область,м.Бердичів,м.Бердичів
312504,9b9b7f43-0b1a-44ee-b0cc-ffa8215408cf,2007,чоловіча,Черкаська область,Уманський район,с.Христинівка,Випускник загальноосвітнього навчального закла...,"селище, село",Христинівський ліцей Христинівської міської ра...,ліцей,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Черкаська область,Уманський район,м.Христинівка
312505,9bc7e9fe-dc70-4cdc-bb3f-ffadc91dc9a2,1986,чоловіча,Полтавська область,м.Полтава,Київський район міста,Випускник минулих років,місто,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Полтавська область,"м.Полтава, Київський район міста",Київський район міста
312506,9b9d9e51-0a61-4b08-b4af-ffb99d790fe5,2000,жіноча,Житомирська область,м.Житомир,Корольовський район міста,Випускник минулих років,місто,NaN,NaN,...,NaN,NaN,NaN,Українська література,Зараховано,"131,0",13.0,Житомирська область,"м.Житомир, Корольовський район міста",Корольовський район міста


In [59]:
dataset_2025 = pd.read_csv(f"../../data_loader/2025/Odata2025File.csv", sep=";", encoding='utf-8')
dataset_2025

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_47754/4078391422.py:1: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset_2025 = pd.read_csv(f"../../data_loader/2025/Odata2025File.csv", sep=";", encoding='utf-8')


,outid,Birth,SexTypeName,RegName,AreaName,TerName,RegTypeName,TerTypeName,EOName,EOTypeName,...,SpaBlockStatus,SpaBlockBall100,SpaBlockBall,UkrLitBlock,UkrLitBlockStatus,UkrLitBlockBall100,UkrLitBlockBall,PTRegName,PTAreaName,PTTerName
0,9e78ac3d-e137-41a2-83df-000bc5e7abd8,2007,чоловіча,Київська область,Броварський район,м.Бровари,Випускник минулих років,місто,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Київська область,м.Бровари,м.Бровари
1,9e77fd9e-00f4-4399-b239-000d421551cc,2008,жіноча,Чернівецька область,Вижницький район,с.Нижні Станівці,Випускник поточного року,"селище, село",Нижньостанівецький заклад загальної середньої ...,середня загальноосвітня школа,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Чернівецька область,Вижницький район,м.Вижниця
2,9e75f45e-7262-4fb8-a347-003a0ea7f2ed,1987,чоловіча,Запорізька область,м.Запоріжжя,Комунарський район міста,Випускник минулих років,місто,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Запорізька область,"м.Запоріжжя, Комунарський район міста",Комунарський район міста
3,9e6812b6-6191-46d3-9048-003ad300b80c,2007,чоловіча,Харківська область,м.Харків,Основ'янський район міста,Випускник поточного року,місто,"ПРИВАТНИЙ ЛІЦЕЙ ""ОНЛАЙН-ШКОЛА ""АЛЬТЕРНАТИВА"" Х...",ліцей,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Чехія,м.Брно,м.Брно
4,9e66bc72-ff85-4864-8a16-002816ca7f73,2008,чоловіча,Харківська область,Куп'янський район,с-ще Шевченкове (Шевченківська),Випускник поточного року,"селище, село",Шевченківський ліцей Шевченківської селищної р...,ліцей,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Словаччина,м.Братислава,м.Братислава
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317086,9e703a35-81af-47bc-b4a9-ff5c468c3756,2008,жіноча,м.Київ,м.Київ,Голосіївський район міста,Випускник поточного року,місто,Ліцей № 59 міста Києва,ліцей,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,м.Київ,"м.Київ, Голосіївський район міста",Голосіївський район міста
317087,9e67ab46-0359-4ae7-9996-ff970ac16a6c,2007,чоловіча,Полтавська область,Кременчуцький район,м.Горішні Плавні,Випускник поточного року,місто,"Відокремлений структурний підрозділ ""Політехні...",заклад фахової передвищої освіти,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Полтавська область,м.Горішні Плавні,м.Горішні Плавні
317088,9e8606ee-f67b-4b37-b01a-ffa775e99884,2002,жіноча,Закарпатська область,Берегівський район,м.Берегове,Випускник минулих років,місто,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Закарпатська область,м.Берегове,м.Берегове
317089,9e77b512-e760-4796-9e7d-ffa8998f2e1a,2008,чоловіча,Одеська область,Березівський район,с-ще Радісне,Випускник поточного року,"селище, село",Радісненський ліцей Знам'янської сільської ради,середня загальноосвітня школа,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Одеська область,Березівський район,с-ще Іванівка (Іванівська)


In [60]:
dataset_2022.columns = [col.lower() for col in dataset_2022.columns]
dataset_2023.columns = [col.lower() for col in dataset_2023.columns]
dataset_2024.columns = [col.lower() for col in dataset_2024.columns]
dataset_2025.columns = [col.lower() for col in dataset_2025.columns]

In [61]:
print([col for col in dataset_2022.columns if col.startswith('pt')])
print([col for col in dataset_2023.columns if col.startswith('pt')])
print([col for col in dataset_2024.columns if col.startswith('pt')])
print([col for col in dataset_2025.columns if col.startswith('pt')])

['ptregname', 'ptareaname', 'pttername']
['ptregname', 'ptareaname', 'pttername']
['ptregname', 'ptareaname', 'pttername']
['ptregname', 'ptareaname', 'pttername']


In [62]:
types = ['', 'eo', 'pt']

In [63]:
locations_KATOTTG

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230,с.Терпіння,23230,85101,2323085101,UA23080270010078454,С
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227,м.Красилів,68227,10100,6822710100,UA68040210010032567,М
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238,с.Дмитрівка,12238,81501,1223881501,UA12140170040016918,С
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101,м.Чернівці,73101,00000,7310100000,UA73060610010033137,М
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223,с.Кумарі,48223,83001,4822383001,UA48080050190079797,С
...,...,...,...,...,...,...,...,...,...,...,...,...
22692,Житомирська область,Любарський район,с.Великий Браталів,18,Любарський район,231,с.Великий Браталів,18231,55106,1823155106,UA18040290070047586,С
22693,Хмельницька область,Ізяславський район,с.Влашанівка,68,Ізяславський район,221,с.Влашанівка,68221,86602,6822186602,UA68060250040099385,С
22694,Львівська область,Яворівський район,с.Коханівка,46,Яворівський район,258,с.Коханівка,46258,87803,4625887803,UA46140110360079694,С
22695,Житомирська область,Ємільчинський район,с.Бастова Рудня,18,Ємільчинський район,217,с.Бастова Рудня,18217,80403,1821780403,UA18080030040080438,С


In [64]:
# OPTIMIZATION: Collect all dataframes first, then concat once
dataframes_to_concat = []

for typ in types:
    for dataset in [dataset_2022, dataset_2023, dataset_2024, dataset_2025]:
        new_col = {typ+'regname': 'regname', typ+'areaname':'areaname', typ+'tername': 'tername'}
        if all(column in dataset.columns for column in new_col.keys()):
            temp_data = dataset.loc[:, new_col.keys()].copy()
            temp_data.rename(columns=new_col, inplace=True)
            # Add NaN columns to match structure
            temp_data.loc[:, ['region_KOATUU', 'areaname_new',
                             'area_KOATUU', 'tername_new', 'code', 'local_KOATUU', 'KOATUU',
                             'KATOTTG', 'category']] = np.nan
            # Drop rows where all original columns are NaN before adding to list
            temp_data = temp_data.dropna(how='all')
            dataframes_to_concat.append(temp_data)

# Single concat operation instead of many
if dataframes_to_concat:
    locations_KATOTTG25 = pd.concat(dataframes_to_concat, ignore_index=True)
    locations_KATOTTG25.drop_duplicates(inplace=True)
    locations_KATOTTG25.reset_index(drop=True, inplace=True)
else:
    locations_KATOTTG25 = pd.DataFrame(columns=locations_KATOTTG.columns)

locations_KATOTTG25

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category
0,Львівська область,Яворівський район,с.Гусаків,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Львівська область,Яворівський район,с.Баличі,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Львівська область,Яворівський район,с.Шегині,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Львівська область,Яворівський район,с.Волиця,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9755,Нідерланди,м.Гаага,м.Гаага,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9756,США,м.Воррен,м.Воррен,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
code2025 = pd.read_excel('./location_data/Codifiers2025.xlsx',  dtype=str, skiprows=3,skipfooter=3)
code2025

,Перший рівень,Другий рівень,Третій рівень,Четвертий рівень,Додатковий рівень,Категорія об’єкта,Назва об’єкта
0,UA01000000000013043,NaN,NaN,NaN,NaN,O,Автономна Республіка Крим
1,UA01000000000013043,UA01020000000022387,NaN,NaN,NaN,P,Бахчисарайський
2,UA01000000000013043,UA01020000000022387,UA01020010000048857,NaN,NaN,H,Андріївська
3,UA01000000000013043,UA01020000000022387,UA01020010000048857,UA01020010010075540,NaN,C,Андріївка
4,UA01000000000013043,UA01020000000022387,UA01020010000048857,UA01020010020030666,NaN,X,Сонячний
...,...,...,...,...,...,...,...
31739,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000875983,B,Святошинський
31740,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000980793,B,Солом’янський
31741,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000001078669,B,Шевченківський
31742,UA85000000000065278,NaN,NaN,NaN,NaN,K,Севастополь


In [66]:
code2025.rename(columns = {'Перший рівень':'first_level','Другий рівень':'second_level','Третій рівень':'third_level', 'Четвертий рівень':'fourth_level',
       'Додатковий рівень':'additional_level', 'Категорія об’єкта':'category','Назва об’єкта':'name'}, inplace=True)
code2025

,first_level,second_level,third_level,fourth_level,additional_level,category,name
0,UA01000000000013043,NaN,NaN,NaN,NaN,O,Автономна Республіка Крим
1,UA01000000000013043,UA01020000000022387,NaN,NaN,NaN,P,Бахчисарайський
2,UA01000000000013043,UA01020000000022387,UA01020010000048857,NaN,NaN,H,Андріївська
3,UA01000000000013043,UA01020000000022387,UA01020010000048857,UA01020010010075540,NaN,C,Андріївка
4,UA01000000000013043,UA01020000000022387,UA01020010000048857,UA01020010020030666,NaN,X,Сонячний
...,...,...,...,...,...,...,...
31739,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000875983,B,Святошинський
31740,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000980793,B,Солом’янський
31741,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000001078669,B,Шевченківський
31742,UA85000000000065278,NaN,NaN,NaN,NaN,K,Севастополь


In [67]:
code2025['name'] = code2025['name'].str.replace('’', "'")

In [68]:
code2025['name_KOATUU']= code2025['name'] 
code2025['name_KOATUU']= code2025['name_KOATUU'].str.strip()
code2025

,first_level,second_level,third_level,fourth_level,additional_level,category,name,name_KOATUU
0,UA01000000000013043,NaN,NaN,NaN,NaN,O,Автономна Республіка Крим,Автономна Республіка Крим
1,UA01000000000013043,UA01020000000022387,NaN,NaN,NaN,P,Бахчисарайський,Бахчисарайський
2,UA01000000000013043,UA01020000000022387,UA01020010000048857,NaN,NaN,H,Андріївська,Андріївська
3,UA01000000000013043,UA01020000000022387,UA01020010000048857,UA01020010010075540,NaN,C,Андріївка,Андріївка
4,UA01000000000013043,UA01020000000022387,UA01020010000048857,UA01020010020030666,NaN,X,Сонячний,Сонячний
...,...,...,...,...,...,...,...,...
31739,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000875983,B,Святошинський,Святошинський
31740,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000980793,B,Солом'янський,Солом'янський
31741,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000000093317,UA80000000001078669,B,Шевченківський,Шевченківський
31742,UA85000000000065278,NaN,NaN,NaN,NaN,K,Севастополь,Севастополь


In [69]:
code2025 = code2025[~code2025['first_level'].str.startswith(('UA01', 'UA85'))]

In [70]:
code2025.category.unique()

array(['O', 'P', 'H', 'C', 'M', 'X', 'B', 'K'], dtype=object)

In [71]:
code2025.loc[code2025['category'] == 'O', 'name_KOATUU'] = code2025.loc[code2025['category'] == 'O', 'name_KOATUU']+' область'
code2025.loc[code2025['category'] == 'P', 'name_KOATUU'] = code2025.loc[code2025['category'] =='P', 'name_KOATUU']+' район'
code2025.loc[code2025['category'] == 'Н', 'name_KOATUU'] = code2025.loc[code2025['category'] == 'Н', 'name_KOATUU']+' громада'
code2025.loc[code2025['category'] == 'C', 'name_KOATUU'] = 'с.' + code2025.loc[code2025['category'] == 'C', 'name_KOATUU']
code2025.loc[code2025['category'] == 'M', 'name_KOATUU'] = 'м.' + code2025.loc[code2025['category'] == 'M', 'name_KOATUU']
code2025.loc[code2025['category'] == 'X', 'name_KOATUU'] = 'с-ще ' + code2025.loc[code2025['category'] == 'X', 'name_KOATUU']
code2025.loc[code2025['category'] == 'В', 'name_KOATUU'] = code2025.loc[code2025['category'] =='В', 'name_KOATUU']+' район в місті'
code2025.loc[code2025['category'] == 'K', 'name_KOATUU'] = 'м.'  + code2025.loc[code2025['category'] == 'K', 'name_KOATUU']

In [72]:
diff = set(code2025[code2025.category.isin(['O','K'])].name_KOATUU.str.lower())^set(locations_KATOTTG25.regname.str.lower())
print(diff)

{'португалія', 'ірландія', 'німеччина', 'швеція', 'молдова', 'угорщина', 'велика британія', 'болгарія', 'швейцарія', 'люксембург', 'латвія', 'нідерланди', 'хорватія', 'словаччина', 'канада', 'фінляндія', 'румунія', 'азербайджан', 'інші країни', 'франція', 'сша', 'естонія', 'бельгія', 'словенія', 'іспанія', 'туреччина', 'польща', 'норвегія', 'данія', 'австрія', 'литва', 'італія', 'чехія', 'грузія'}


In [73]:
locations_KATOTTG25['region_KATOTTG'] = np.nan

In [74]:
for reg in code2025[code2025.category.isin(['O','K'])].name_KOATUU.str.lower().unique():
    values = code2025.loc[code2025.name_KOATUU.str.lower() == reg, 'first_level'].unique()
    if len(values) == 1:
        locations_KATOTTG25.loc[(locations_KATOTTG25.regname.str.lower()==reg), 'region_KATOTTG'] = values[0]
    else:
        raise Exception(f'More than one value:{reg}, {values}')

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_47754/59978042.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'UA05000000000010236' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  locations_KATOTTG25.loc[(locations_KATOTTG25.regname.str.lower()==reg), 'region_KATOTTG'] = values[0]


In [75]:
locations_KATOTTG25[~locations_KATOTTG25.regname.str.lower().isin(diff)].region_KATOTTG.unique()

array(['UA46000000000026241', 'UA26000000000069363',
       'UA07000000000024379', 'UA51000000000030770',
       'UA35000000000016081', 'UA56000000000066151',
       'UA48000000000039575', 'UA32000000000030281',
       'UA21000000000011690', 'UA05000000000010236',
       'UA80000000000093317', 'UA14000000000091971',
       'UA65000000000030969', 'UA18000000000041385',
       'UA23000000000064947', 'UA12000000000090473',
       'UA61000000000060328', 'UA73000000000044923',
       'UA74000000000025378', 'UA53000000000028050',
       'UA71000000000010357', 'UA63000000000041885',
       'UA44000000000018893', 'UA59000000000057109',
       'UA68000000000099709'], dtype=object)

In [76]:
locations_KATOTTG25.loc[(locations_KATOTTG25.regname.str.lower().isin(diff)), 'region_KATOTTG'] ='OC00000000000000000'
locations_KATOTTG25

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG
0,Львівська область,Яворівський район,с.Гусаків,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241
1,Львівська область,Яворівський район,с.Баличі,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241
2,Львівська область,Яворівський район,с.Шегині,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241
3,Львівська область,Яворівський район,с.Волиця,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA26000000000069363
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA51000000000030770
9755,Нідерланди,м.Гаага,м.Гаага,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000
9756,США,м.Воррен,м.Воррен,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000


In [77]:
locations_KATOTTG25.loc[(locations_KATOTTG25.region_KATOTTG.str.startswith('OC')), 'region_new'] ='Інші країни'
locations_KATOTTG25.loc[(locations_KATOTTG25.region_KATOTTG.str.startswith('UA')), 'region_new'] = locations_KATOTTG25.regname
locations_KATOTTG25

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new
0,Львівська область,Яворівський район,с.Гусаків,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область
1,Львівська область,Яворівський район,с.Баличі,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область
2,Львівська область,Яворівський район,с.Шегині,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область
3,Львівська область,Яворівський район,с.Волиця,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA26000000000069363,Івано-Франківська область
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA51000000000030770,Одеська область
9755,Нідерланди,м.Гаага,м.Гаага,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9756,США,м.Воррен,м.Воррен,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни


In [78]:
locations_KATOTTG25[locations_KATOTTG25.region_KATOTTG.isna()]

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new


# Areaname 2022

заповнити areaname_new аналогічно попереднього коду

In [79]:
locations_KATOTTG25['areaname_new'] = locations_KATOTTG25['areaname']

In [80]:
locations_KATOTTG25['areaname_new'].unique()

array(['Яворівський район', 'Надвірнянський район', 'Луцький район',
       'Івано-Франківський район', 'м.Одеса', 'м.Львів',
       'м.Кропивницький', 'Рівненський район', 'Первомайський район',
       'Бориспільський район', 'Берегівський район', 'Вінницький район',
       'Самбірський район', 'Ужгородський район', 'м.Київ',
       'Волноваський район', 'Скадовський район', 'Житомирський район',
       'Мелітопольський район', 'Львівський район', 'Гайсинський район',
       'Ізмаїльський район', 'Краматорський район', 'Криворізький район',
       'Чортківський район', 'Вижницький район', 'Прилуцький район',
       'Вишгородський район', 'Чернівецький район', 'Лубенський район',
       'Сарненський район', 'м.Черкаси', "Кам'янський район",
       'Дубенський район', 'Тернопільський район', 'Дніпровський район',
       'Пологівський район', 'Вознесенський район', 'Хмільницький район',
       'Уманський район', 'Каховський район', 'Володимирський район',
       'Мукачівський район', 'Ко

In [81]:
locations_KATOTTG25['areaname_new'] = locations_KATOTTG25.apply(lambda row: row['tername'].split()[0] if ('м.Київ' in row['areaname_new']) else row['areaname_new'], axis=1)

In [82]:
locations_KATOTTG25['areaname_new'] = locations_KATOTTG25.apply(lambda row: row['areaname_new'].split(', ')[0].replace(' район міста','') if (', ' in row['areaname_new']) else row['areaname_new'], axis=1)

In [83]:
diff.remove('інші країни')

In [84]:
diff

{'австрія',
 'азербайджан',
 'бельгія',
 'болгарія',
 'велика британія',
 'грузія',
 'данія',
 'естонія',
 'канада',
 'латвія',
 'литва',
 'люксембург',
 'молдова',
 'норвегія',
 'нідерланди',
 'німеччина',
 'польща',
 'португалія',
 'румунія',
 'словаччина',
 'словенія',
 'сша',
 'туреччина',
 'угорщина',
 'франція',
 'фінляндія',
 'хорватія',
 'чехія',
 'швейцарія',
 'швеція',
 'ірландія',
 'іспанія',
 'італія'}

In [85]:
locations_KATOTTG25

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA26000000000069363,Івано-Франківська область
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA51000000000030770,Одеська область
9755,Нідерланди,м.Гаага,м.Гаага,NaN,м.Гаага,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9756,США,м.Воррен,м.Воррен,NaN,м.Воррен,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,м.Домброва-Гурнича,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни


In [86]:
locations_KATOTTG25[locations_KATOTTG25.regname.str.lower() == 'інші країни']

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new
8161,Інші країни,Польща,м.Варшава,NaN,Польща,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
8163,Інші країни,Франція,м.Париж,NaN,Франція,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
8165,Інші країни,Німеччина,м.Берлін,NaN,Німеччина,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
8167,Інші країни,Німеччина,м.Мюнхен,NaN,Німеччина,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
8168,Інші країни,Польща,м.Люблін,NaN,Польща,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9673,Інші країни,м.Гронінген,м.Гронінген,NaN,м.Гронінген,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9674,Інші країни,м.Люксембург,м.Люксембург,NaN,м.Люксембург,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9675,Інші країни,м.Марібор,м.Марібор,NaN,м.Марібор,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9676,Інші країни,м.Сакраменто,м.Сакраменто,NaN,м.Сакраменто,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни


In [87]:
locations_KATOTTG25['areaname_new'] = locations_KATOTTG25.apply(lambda row: row['regname'] if (row['regname'].lower() in diff) else row['areaname_new'], axis=1)
locations_KATOTTG25.tail()

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA51000000000030770,Одеська область
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни
9758,Туреччина,м.Стамбул,м.Стамбул,NaN,Туреччина,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни


In [88]:
code2025[(code2025.first_level == 'UA18000000000041385')&(code2025.category.isin(['P','B', 'M']))]

,first_level,second_level,third_level,fourth_level,additional_level,category,name,name_KOATUU
7092,UA18000000000041385,UA18020000000072859,NaN,NaN,NaN,P,Бердичівський,Бердичівський район
7094,UA18000000000041385,UA18020000000072859,UA18020010000059904,UA18020010010097402,NaN,M,Андрушівка,м.Андрушівка
7120,UA18000000000041385,UA18020000000072859,UA18020030000035625,UA18020030010047029,NaN,M,Бердичів,м.Бердичів
7265,UA18000000000041385,UA18040000000058965,NaN,NaN,NaN,P,Житомирський,Житомирський район
7383,UA18000000000041385,UA18040000000058965,UA18040190000029215,UA18040190010057814,NaN,M,Житомир,м.Житомир
7384,UA18000000000041385,UA18040000000058965,UA18040190000029215,UA18040190010057814,UA18040190010115253,B,Богунський,Богунський
7385,UA18000000000041385,UA18040000000058965,UA18040190000029215,UA18040190010057814,UA18040190010281147,B,Корольовський,Корольовський
7409,UA18000000000041385,UA18040000000058965,UA18040250000068141,UA18040250010085012,NaN,M,Коростишів,м.Коростишів
7681,UA18000000000041385,UA18040000000058965,UA18040450000079138,UA18040450010049683,NaN,M,Радомишль,м.Радомишль
7948,UA18000000000041385,UA18040000000058965,UA18040610000049084,UA18040610010066897,NaN,M,Чуднів,м.Чуднів


In [89]:
importlib.reload(renam)

<module 'src.renaming_dictionaries' from '/Users/scipyguru/Library/Mobile Documents/com~apple~CloudDocs/Documents_New/ZNO-Dataset/notebooks/tables_creation/src/renaming_dictionaries.py'>

In [90]:
for code in renam.dct_rename_area_2023:
    for name in renam.dct_rename_area_2023[code]:
        locations_KATOTTG25.loc[(locations_KATOTTG25.region_KATOTTG == code) & (locations_KATOTTG25['areaname_new'].str.lower() == name.lower()), 'areaname_new'] = renam.dct_rename_area_2023[code][name]

In [91]:
troubles = {}
for reg in locations_KATOTTG25.region_KATOTTG.unique():
    trouble = set(locations_KATOTTG25[(locations_KATOTTG25.region_KATOTTG == reg)].areaname_new.str.lower())-set(code2025[(code2025.first_level == reg)&(code2025.category.isin(['P','B', 'M']))].name_KOATUU.str.lower())
    if trouble:
        troubles[reg] = trouble
troubles

{'OC00000000000000000': {'австрія',
  'азербайджан',
  'бельгія',
  'болгарія',
  'велика британія',
  'греція',
  'грузія',
  'данія',
  'естонія',
  'канада',
  'латвія',
  'литва',
  'люксембург',
  'молдова',
  'норвегія',
  'нідерланди',
  'німеччина',
  'польща',
  'португалія',
  'румунія',
  'словаччина',
  'словенія',
  'сша',
  'туреччина',
  'угорщина',
  'франція',
  'фінляндія',
  'хорватія',
  'чехія',
  'швейцарія',
  'швеція',
  'ірландія',
  'іспанія',
  'італія'}}

In [92]:
locations_KATOTTG25['area_KATOTTG'] = np.nan
locations_KATOTTG25

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,NaN
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,NaN
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,NaN
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,NaN
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA26000000000069363,Івано-Франківська область,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA51000000000030770,Одеська область,NaN
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни,NaN
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни,NaN
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни,NaN


In [93]:
for reg in locations_KATOTTG25['region_KATOTTG'].unique():
    reg_filtered_data = locations_KATOTTG25[locations_KATOTTG25['region_KATOTTG'] == reg]
    
    for area in reg_filtered_data['areaname_new'].str.lower().unique():
        values = code2025.loc[(code2025['first_level'] == reg) & (code2025['name_KOATUU'].str.lower() == area), 'category'].unique()
        if len(values) == 0:
            print(f'Not found for {reg}, {area}: {values}')
        elif len(values) == 1:
            if values[0] in ['P', 'M']:
                value2 = code2025.loc[(code2025['first_level'] == reg) & (code2025['name_KOATUU'].str.lower() == area), 'second_level'].unique()
                
                if len(value2) == 1:
                    locations_KATOTTG25.loc[(locations_KATOTTG25['region_KATOTTG'] == reg) & (locations_KATOTTG25['areaname_new'].str.lower() == area), 'area_KATOTTG'] = value2[0]
                else:
                    print(f"Warning: More than one value for {reg}, {area}, {value2}")
            elif values[0] == 'B':
                value_add = code2025.loc[(code2025['first_level'] == reg) & (code2025['name_KOATUU'].str.lower() == area), 'additional_level'].unique()
                
                if len(value_add) == 1:
                    locations_KATOTTG25.loc[(locations_KATOTTG25['region_KATOTTG'] == reg)  & (locations_KATOTTG25['areaname_new'].str.lower() == area), 'area_KATOTTG'] = value_add[0]
                else:
                    print(f"Warning: More than one value for {reg}, {area}, {value_add}")
        else:
            print(f"Warning: More than one value for {reg}, {area}, {values}")


/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_47754/630345596.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'UA46140000000036328' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  locations_KATOTTG25.loc[(locations_KATOTTG25['region_KATOTTG'] == reg) & (locations_KATOTTG25['areaname_new'].str.lower() == area), 'area_KATOTTG'] = value2[0]


Not found for OC00000000000000000, польща: []
Not found for OC00000000000000000, франція: []
Not found for OC00000000000000000, німеччина: []
Not found for OC00000000000000000, естонія: []
Not found for OC00000000000000000, норвегія: []
Not found for OC00000000000000000, словаччина: []
Not found for OC00000000000000000, австрія: []
Not found for OC00000000000000000, велика британія: []
Not found for OC00000000000000000, ірландія: []
Not found for OC00000000000000000, болгарія: []
Not found for OC00000000000000000, молдова: []
Not found for OC00000000000000000, португалія: []
Not found for OC00000000000000000, азербайджан: []
Not found for OC00000000000000000, угорщина: []
Not found for OC00000000000000000, італія: []
Not found for OC00000000000000000, туреччина: []
Not found for OC00000000000000000, чехія: []
Not found for OC00000000000000000, бельгія: []
Not found for OC00000000000000000, іспанія: []
Not found for OC00000000000000000, канада: []
Not found for OC00000000000000000, хорв

In [94]:
for name in renam.dct_code_area_2023:
    locations_KATOTTG25.loc[(locations_KATOTTG25.region_KATOTTG == 'OC00000000000000000') & (locations_KATOTTG25['areaname_new'].str.lower() == name.lower()), 'area_KATOTTG'] = renam.dct_code_area_2023[name]

In [95]:
locations_KATOTTG25[locations_KATOTTG25.area_KATOTTG.isna()]

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG


# tername 2022

In [96]:
locations_KATOTTG25

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,UA46140000000036328
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,UA46140000000036328
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,UA46140000000036328
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,UA46140000000036328
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA26000000000069363,Івано-Франківська область,UA26120000000036058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UA51000000000030770,Одеська область,UA51100000000095786
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни,OC18000000000000000
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни,OC26000000000000000
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни,OC21000000000000000


In [97]:
locations_KATOTTG25['tername_new'] = locations_KATOTTG25['tername']
locations_KATOTTG25['tername_new'] = locations_KATOTTG25['tername_new'].str.replace(r'\(.*\)', '', regex = True).str.strip()
locations_KATOTTG25['tername_new'] = locations_KATOTTG25['tername_new'].str.replace(' район міста','')
locations_KATOTTG25['tername_new'] = locations_KATOTTG25['tername_new'].str.replace('смт','с-ще')

In [98]:
locations_KATOTTG25

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,с.Гусаків,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,UA46140000000036328
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,с.Баличі,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,UA46140000000036328
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,с.Шегині,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,UA46140000000036328
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,с.Волиця,NaN,NaN,NaN,NaN,NaN,UA46000000000026241,Львівська область,UA46140000000036328
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,с.Гвізд,NaN,NaN,NaN,NaN,NaN,UA26000000000069363,Івано-Франківська область,UA26120000000036058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,м.Південне,NaN,NaN,NaN,NaN,NaN,UA51000000000030770,Одеська область,UA51100000000095786
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,м.Гаага,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни,OC18000000000000000
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,м.Воррен,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни,OC26000000000000000
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,м.Домброва-Гурнича,NaN,NaN,NaN,NaN,NaN,OC00000000000000000,Інші країни,OC21000000000000000


In [99]:
importlib.reload(renam)

<module 'src.renaming_dictionaries' from '/Users/scipyguru/Library/Mobile Documents/com~apple~CloudDocs/Documents_New/ZNO-Dataset/notebooks/tables_creation/src/renaming_dictionaries.py'>

In [100]:
for reg in locations_KATOTTG25.region_KATOTTG.unique():
    troubles = {}
    for area in locations_KATOTTG25[(locations_KATOTTG25.region_KATOTTG == reg)].area_KATOTTG.unique():
        trouble = set(locations_KATOTTG25[(locations_KATOTTG25.region_KATOTTG == reg)&(locations_KATOTTG25.area_KATOTTG == area)].tername_new.str.lower())-set(code2025[(code2025.first_level == reg)&((code2025.second_level == area)|(code2025.additional_level == area))&(code2025.category.isin(['C', 'M', 'T', 'X', 'B']))].name_KOATUU.str.lower())
        if trouble:
            troubles[area] = trouble
    if troubles:
        print(f'{repr(reg)}:{troubles},')

'UA46000000000026241':{'UA46140000000036328': {'с.іорданівка'}, 'UA46060000000042587': {'с-ще рудне', 'с.милошевичі'}, 'UA46100000000023056': {'с-ще славське', 'с.кавське'}, 'UA46040000000069196': {'с.підгородне', 'с.червоне'}, 'UA46120000000057093': {'м.червоноград'}},
'UA26000000000069363':{'UA26080000000050130': {'с.воскресинці', 'с.троїця'}},
'UA07000000000024379':{'UA07080000000034745': {'с-ще олика'}, 'UA07020000000048211': {'с.петрове'}},
'UA51000000000030770':{'UA51100000000095786': {'м.южне', 'с.першотравневе', 'с.гвардійське', 'малиновський', 'суворовський'}, 'UA51080000000061776': {'с.першотравневе', 'с-ще суворове'}, 'UA51020000000097683': {'с.андрієво-іванівка', 'с-ще петрівка'}, 'UA51060000000057808': {'с-ще бородіно', 'с.надеждівка', 'с.малоярославець перший', 'с.холмське', 'с-ще березине', 'с.малоярославець другий', 'с-ще тарутине'}, 'UA51120000000021678': {'с.ткаченка'}, 'UA51040000000032911': {'с.надежда', 'с.миколаївка-новоросійська', 'с.зоря'}, 'UA51140000000094970'

In [101]:
for region, next_info in renam.dct_rename_tername_2025.items():
    for area, names in next_info.items():
        for old_name, new_name in names.items():
            locations_KATOTTG25.loc[(locations_KATOTTG25.region_KATOTTG == region) & (locations_KATOTTG25.area_KATOTTG== area) & (locations_KATOTTG25.tername_new.str.lower() == old_name), 'tername_new'] = new_name

In [102]:
for reg in locations_KATOTTG25.region_KATOTTG.unique():
    troubles = {}
    for area in locations_KATOTTG25[(locations_KATOTTG25.region_KATOTTG == reg)].area_KATOTTG.unique():
        trouble = set(locations_KATOTTG25[(locations_KATOTTG25.region_KATOTTG == reg)&(locations_KATOTTG25.area_KATOTTG == area)].tername_new.str.lower())-set(code2025[(code2025.first_level == reg)&((code2025.second_level == area)|(code2025.additional_level == area))&(code2025.category.isin(['C', 'M', 'T', 'X', 'B']))].name_KOATUU.str.lower())
        if trouble:
            troubles[area] = trouble
    if troubles:
        print(f'{repr(reg)}:{troubles},')

'OC00000000000000000':{'OC21000000000000000': {'м.бидгощ', 'м.варшава', 'м.краків', 'м.домброва-гурнича', 'м.катовіце', 'м.люблін', 'м.вроцлав', 'м.гданськ', 'м.познань', 'м.домброва горнича'}, 'OC30000000000000000': {'м.бордо', 'м.париж', 'м.марсель'}, 'OC19000000000000000': {'м.мюнхен', 'м.франкфурт-на-майні', 'м.магдебург', 'м.гамбург', 'м.дюссельдорф', 'м.кельн', 'м.пассау', 'м.лейпциг', 'м.берлін', 'м.єна', 'м.нюрнберг', 'м.майнц'}, 'OC12000000000000000': {'м.таллінн', 'м.тарту'}, 'OC20000000000000000': {'м.осло'}, 'OC24000000000000000': {'м.братислава', 'м.банська бистриця'}, 'OC05000000000000000': {'м.відень'}, 'OC08000000000000000': {'м.лондон', 'м.единбург', 'м.абериствіт'}, 'OC01000000000000000': {'м.тралі', 'м.дублін'}, 'OC07000000000000000': {'м.варна', 'м.софія'}, 'OC17000000000000000': {'м.кишинів'}, 'OC22000000000000000': {'м.лісабон'}, 'OC04000000000000000': {'м.баку'}, 'OC28000000000000000': {'м.дьєр', 'м.будапешт', 'м.дебрецен'}, 'OC03000000000000000': {'м.мілан', 'м.

In [103]:
locations_KATOTTG25['third_level'] = locations_KATOTTG25.loc[:, 'tername'].str.extract(r'\((.*?)\)')
locations_KATOTTG25[locations_KATOTTG25.third_level.notna()]


,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG,third_level
39,Запорізька область,Мелітопольський район,с.Розівка (Якимівська),NaN,Мелітопольський район,NaN,с.Розівка,NaN,NaN,NaN,NaN,NaN,UA23000000000064947,Запорізька область,UA23080000000090746,Якимівська
47,Донецька область,Краматорський район,с.Торське (Лиманська),NaN,Краматорський район,NaN,с.Торське,NaN,NaN,NaN,NaN,NaN,UA14000000000091971,Донецька область,UA14120000000070232,Лиманська
70,Вінницька область,Гайсинський район,с.Яланець (Бершадська),NaN,Гайсинський район,NaN,с.Яланець,NaN,NaN,NaN,NaN,NaN,UA05000000000010236,Вінницька область,UA05040000000050292,Бершадська
71,Вінницька область,Гайсинський район,с.Мар'янівка (Гайсинська),NaN,Гайсинський район,NaN,с.Мар'янівка,NaN,NaN,NaN,NaN,NaN,UA05000000000010236,Вінницька область,UA05040000000050292,Гайсинська
94,Тернопільська область,Чортківський район,с.Озеряни (Борщівська),NaN,Чортківський район,NaN,с.Озеряни,NaN,NaN,NaN,NaN,NaN,UA61000000000060328,Тернопільська область,UA61060000000068766,Борщівська
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9363,Харківська область,Берестинський район,с.Миколаївка (Старовірівська),NaN,Берестинський район,NaN,с.Миколаївка,NaN,NaN,NaN,NaN,NaN,UA63000000000041885,Харківська область,UA63060000000053698,Старовірівська
9378,Харківська область,Берестинський район,с.Шевченкове (Сахновщинська),NaN,Берестинський район,NaN,с.Шевченкове,NaN,NaN,NaN,NaN,NaN,UA63000000000041885,Харківська область,UA63060000000053698,Сахновщинська
9380,Чернігівська область,Ніжинський район,с.Григорівка (Бахмацька),NaN,Ніжинський район,NaN,с.Григорівка,NaN,NaN,NaN,NaN,NaN,UA74000000000025378,Чернігівська область,UA74040000000028062,Бахмацька
9385,Харківська область,Ізюмський район,с.Підвисоке (Борівська),NaN,Ізюмський район,NaN,с.Підвисоке,NaN,NaN,NaN,NaN,NaN,UA63000000000041885,Харківська область,UA63040000000023521,Борівська


In [104]:
locations_KATOTTG25['third_level_code'] = locations_KATOTTG25['third_level']

In [105]:
for reg in locations_KATOTTG25['region_KATOTTG'].unique():
    reg_filtered_data = locations_KATOTTG25[(locations_KATOTTG25['region_KATOTTG'] == reg)]
    
    for area in reg_filtered_data['area_KATOTTG'].unique():
        area_filtered_data = reg_filtered_data[(reg_filtered_data['area_KATOTTG'] == area)]
        
        for name in area_filtered_data['third_level'].str.lower().unique():
            values = code2025.loc[
                (code2025['first_level'] == reg) &
                (code2025['second_level'] == area) &
                (code2025['name_KOATUU'].str.lower() == name) &
                (code2025['category'].isin(['H', 'B'])), 'third_level'].unique()
            
            if len(values) == 1:
                locations_KATOTTG25.loc[
                    (locations_KATOTTG25['region_KATOTTG'] == reg) &
                    (locations_KATOTTG25['area_KATOTTG'] == area) &
                    (locations_KATOTTG25['third_level'].str.lower() == name), 'third_level_code'] = values[0]
            elif len(values) > 1:
                codes = locations_KATOTTG25.loc[
                    (locations_KATOTTG25['region_KATOTTG'] == reg) &
                    (locations_KATOTTG25['area_KATOTTG'] == area) &
                    (locations_KATOTTG25['third_level'].str.lower() == name), 'third_level_code'].unique()
                if not all(code.startswith('UA') for code in codes):
                    print(f"Warning: More than one value for {reg}, {area}, {name}, {values}")
            elif len(values) == 0 and isinstance(name, str):
                print(f"No values found for {reg}, {area}, {name}, {values}")


No values found for UA46000000000026241, UA46120000000057093, червоноградська, []
No values found for UA51000000000030770, UA51020000000097683, новокальчевська, []
No values found for UA51000000000030770, UA51060000000057808, тарутинська, []
No values found for UA35000000000016081, UA35040000000034705, новгородківська, []
No values found for UA35000000000016081, UA35060000000086172, мар'янівська, []
No values found for UA48000000000039575, UA48040000000011780, южноукраїнська, []
No values found for UA48000000000039575, UA48060000000094390, радсадівська, []
No values found for UA23000000000064947, UA23080000000090746, чкаловська, []
No values found for UA23000000000064947, UA23020000000032289, коларівська, []
No values found for UA12000000000090473, UA12140000000011720, іларіонівська, []
No values found for UA12000000000090473, UA12140000000011720, брагинівська, []
No values found for UA63000000000041885, UA63140000000096278, чкаловська, []
No values found for UA44000000000018893, UA441

In [106]:
for region, areas in renam.dct_third_level_fixes_2025.items():
    for area, fixes in areas.items():
        for old_name, payload in fixes.items():
            new_name = payload.get('new', old_name)
            mask = (
                (locations_KATOTTG25.region_KATOTTG == region) &
                (locations_KATOTTG25.area_KATOTTG == area) &
                (locations_KATOTTG25.third_level.str.lower() == old_name)
            )
            locations_KATOTTG25.loc[mask, 'third_level'] = new_name
            if 'code' in payload:
                locations_KATOTTG25.loc[mask, 'third_level_code'] = payload['code']

In [107]:
all(code.startswith('UA') for code in locations_KATOTTG25['third_level_code'].unique() if pd.notna(code))

True

In [149]:
locations_2025 =locations_KATOTTG25

In [151]:
locations_2025['third_level_code'].fillna(-1, inplace=True)

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_47754/2594264417.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  locations_2025['third_level_code'].fillna(-1, inplace=True)


In [152]:
locations_2025

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG,third_level,third_level_code
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,с.Гусаків,NaN,NaN,4622481601,UA46140090120020894,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,с.Баличі,NaN,NaN,4622480401,UA46140090020037836,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,с.Шегині,NaN,NaN,4622487901,UA46140090010034963,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,с.Волиця,NaN,NaN,4622480901,UA46140090090088787,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,с.Гвізд,NaN,NaN,2624081501,UA26120070030065363,С,UA26000000000069363,Івано-Франківська область,UA26120000000036058,NaN,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,м.Південне,NaN,NaN,5111700000,UA51100410010044384,М,UA51000000000030770,Одеська область,UA51100000000095786,NaN,-1
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,м.Гаага,NaN,NaN,NaN,OC18030000000000000,NaN,OC00000000000000000,Інші країни,OC18000000000000000,NaN,-1
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,м.Воррен,NaN,NaN,NaN,OC26010000000000000,NaN,OC00000000000000000,Інші країни,OC26000000000000000,NaN,-1
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,м.Домброва-Гурнича,NaN,NaN,NaN,OC21050000000000000,NaN,OC00000000000000000,Інші країни,OC21000000000000000,NaN,-1


In [153]:
for name in renam.dct_code_cities_2025:
   locations_2025.loc[(locations_2025.region_KATOTTG == 'OC00000000000000000') & (locations_2025['tername_new'].str.lower() == name.lower()), 'KATOTTG'] = renam.dct_code_cities_2025[name]

In [154]:
locations_2025

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG,third_level,third_level_code
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,с.Гусаків,NaN,NaN,4622481601,UA46140090120020894,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,с.Баличі,NaN,NaN,4622480401,UA46140090020037836,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,с.Шегині,NaN,NaN,4622487901,UA46140090010034963,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,с.Волиця,NaN,NaN,4622480901,UA46140090090088787,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,с.Гвізд,NaN,NaN,2624081501,UA26120070030065363,С,UA26000000000069363,Івано-Франківська область,UA26120000000036058,NaN,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,м.Південне,NaN,NaN,5111700000,UA51100410010044384,М,UA51000000000030770,Одеська область,UA51100000000095786,NaN,-1
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,м.Гаага,NaN,NaN,NaN,OC18030000000000000,NaN,OC00000000000000000,Інші країни,OC18000000000000000,NaN,-1
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,м.Воррен,NaN,NaN,NaN,OC26010000000000000,NaN,OC00000000000000000,Інші країни,OC26000000000000000,NaN,-1
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,м.Домброва-Гурнича,NaN,NaN,NaN,OC21050000000000000,NaN,OC00000000000000000,Інші країни,OC21000000000000000,NaN,-1


In [155]:
locations_2025[(locations_2025.KATOTTG.isna())&(locations_2025.region_KATOTTG.str.startswith('OC'))][['region_new', 'areaname_new', 'tername_new']].value_counts()

Series([], Name: count, dtype: int64)

In [156]:
# OPTIMIZATION 1: Pre-compute lowercase name_KOATUU once (not in every iteration)
if 'name_KOATUU_lower' not in code2025.columns:
    code2025.loc[:, 'name_KOATUU_lower'] = code2025['name_KOATUU'].str.lower()

# OPTIMIZATION 2: Create filtered subsets once (not in every iteration)
code2025_cmtx = code2025[code2025['category'].isin(['C', 'M', 'T', 'X'])].copy()
code2025_b = code2025[code2025['category'] == 'B'].copy()

# OPTIMIZATION 3: Pre-compute group info to avoid repeated .loc calls
grouped_eie = locations_2025.groupby(['region_KATOTTG', 'area_KATOTTG', 'third_level_code', 'tername_new'])

for (reg, area, third_lev, name), group in grouped_eie:
    # Skip if already has KATOTTG
    if locations_2025.loc[group.index, 'KATOTTG'].notna().any():
        continue
    
    lower_name = name.lower()
    
    if isinstance(third_lev, str):
        # Use pre-filtered dataframes
        mask = (
            (code2025_cmtx['first_level'] == reg) &
            (code2025_cmtx['second_level'] == area) &
            (code2025_cmtx['third_level'] == third_lev) &
            (code2025_cmtx['name_KOATUU_lower'] == lower_name)
        )
        values = code2025_cmtx.loc[mask, 'fourth_level'].unique()
        
        mask_add = (
            (code2025_b['first_level'] == reg) &
            (code2025_b['second_level'] == area) &
            (code2025_b['third_level'] == third_lev) &
            (code2025_b['name_KOATUU_lower'] == lower_name)
        )
        values_add = code2025_b.loc[mask_add, 'additional_level'].unique()
    
    else:
        mask = (
            (code2025_cmtx['first_level'] == reg) &
            (code2025_cmtx['second_level'] == area) &
            (code2025_cmtx['name_KOATUU_lower'] == lower_name)
        )
        values = code2025_cmtx.loc[mask, 'fourth_level'].unique()
        
        if reg != 'UA80000000000093317':
            mask_add = (
                (code2025_b['first_level'] == reg) &
                (code2025_b['second_level'] == area) &
                (code2025_b['name_KOATUU_lower'] == lower_name)
            )
            values_add = code2025_b.loc[mask_add, 'additional_level'].unique()
        else:
            mask_add = (
                (code2025_b['first_level'] == reg) &
                (code2025_b['additional_level'] == area) &
                (code2025_b['name_KOATUU_lower'] == lower_name)
            )
            values_add = code2025_b.loc[mask_add, 'additional_level'].unique()

    if len(values) == 1:
        locations_2025.loc[group.index, 'KATOTTG'] = values[0]
    elif len(values_add) == 1:
        locations_2025.loc[group.index, 'KATOTTG'] = values_add[0]
    else:
        print(reg, area, third_lev, name, values, values_add)

In [115]:
# locations_2025[
#     (locations_2025['region_KATOTTG'] == 'UA63000000000041885') &
#     (locations_2025['area_KATOTTG'] == 'UA63140000000096278') &
#     (locations_2025['tername_new'] == "с.Варварівка")&
#     (locations_2025['KATOTTG'].isna())]

In [157]:
# Manual fixes for tername_new and KATOTTG using dictionary
# якщо є два населених пункти з однаковими назвами, то встановлюємо з більшим населенням
import importlib
importlib.reload(renam)

for region, areas in renam.dct_tername_katottg_fixes_2025.items():
    for area, fixes in areas.items():
        for old_name, payload in fixes.items():
            # If 'new_tername' is present, match against original 'tername' and update both tername_new and KATOTTG
            if 'new_tername' in payload:
                mask = (
                    (locations_KATOTTG25.region_KATOTTG == region) &
                    (locations_KATOTTG25.area_KATOTTG == area) &
                    (locations_KATOTTG25.tername.str.lower() == old_name)
                )
                locations_KATOTTG25.loc[mask, 'tername_new'] = payload['new_tername']
                if 'katottg' in payload:
                    locations_KATOTTG25.loc[mask, 'KATOTTG'] = payload['katottg']
            else:
                # Otherwise, match against 'tername_new' and set KATOTTG
                mask = (
                    (locations_KATOTTG25.region_KATOTTG == region) &
                    (locations_KATOTTG25.area_KATOTTG == area) &
                    (locations_KATOTTG25.tername_new.str.lower() == old_name)
                )
                if 'katottg' in payload:
                    locations_KATOTTG25.loc[mask, 'KATOTTG'] = payload['katottg']

In [158]:
locations_2025

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG,third_level,third_level_code
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,с.Гусаків,NaN,NaN,4622481601,UA46140090120020894,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,с.Баличі,NaN,NaN,4622480401,UA46140090020037836,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,с.Шегині,NaN,NaN,4622487901,UA46140090010034963,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,с.Волиця,NaN,NaN,4622480901,UA46140090090088787,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,с.Гвізд,NaN,NaN,2624081501,UA26120070030065363,С,UA26000000000069363,Івано-Франківська область,UA26120000000036058,NaN,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,м.Південне,NaN,NaN,5111700000,UA51100410010044384,М,UA51000000000030770,Одеська область,UA51100000000095786,NaN,-1
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,м.Гаага,NaN,NaN,NaN,OC18030000000000000,NaN,OC00000000000000000,Інші країни,OC18000000000000000,NaN,-1
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,м.Воррен,NaN,NaN,NaN,OC26010000000000000,NaN,OC00000000000000000,Інші країни,OC26000000000000000,NaN,-1
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,м.Домброва-Гурнича,NaN,NaN,NaN,OC21050000000000000,NaN,OC00000000000000000,Інші країни,OC21000000000000000,NaN,-1


In [159]:
locations_2025[(locations_2025.KATOTTG.isna())]

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG,third_level,third_level_code


In [160]:
locations_2025

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG,third_level,third_level_code
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,с.Гусаків,NaN,NaN,4622481601,UA46140090120020894,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,с.Баличі,NaN,NaN,4622480401,UA46140090020037836,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,с.Шегині,NaN,NaN,4622487901,UA46140090010034963,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,с.Волиця,NaN,NaN,4622480901,UA46140090090088787,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,с.Гвізд,NaN,NaN,2624081501,UA26120070030065363,С,UA26000000000069363,Івано-Франківська область,UA26120000000036058,NaN,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,м.Південне,NaN,NaN,5111700000,UA51100410010044384,М,UA51000000000030770,Одеська область,UA51100000000095786,NaN,-1
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,м.Гаага,NaN,NaN,NaN,OC18030000000000000,NaN,OC00000000000000000,Інші країни,OC18000000000000000,NaN,-1
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,м.Воррен,NaN,NaN,NaN,OC26010000000000000,NaN,OC00000000000000000,Інші країни,OC26000000000000000,NaN,-1
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,м.Домброва-Гурнича,NaN,NaN,NaN,OC21050000000000000,NaN,OC00000000000000000,Інші країни,OC21000000000000000,NaN,-1


In [161]:
locations_2025['KOATUU'] = locations_2025['KOATUU'].fillna(locations_2025['KATOTTG'].map(comparison.set_index('KATOTTG')['KOATUU']))

In [162]:
locations_2025['category'] = locations_2025['category'].fillna(locations_2025['KATOTTG'].map(comparison.set_index('KATOTTG')['category']))
locations_2025

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG,third_level,third_level_code
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,с.Гусаків,NaN,NaN,4622481601,UA46140090120020894,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,с.Баличі,NaN,NaN,4622480401,UA46140090020037836,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,с.Шегині,NaN,NaN,4622487901,UA46140090010034963,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,с.Волиця,NaN,NaN,4622480901,UA46140090090088787,С,UA46000000000026241,Львівська область,UA46140000000036328,NaN,-1
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,с.Гвізд,NaN,NaN,2624081501,UA26120070030065363,С,UA26000000000069363,Івано-Франківська область,UA26120000000036058,NaN,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,м.Південне,NaN,NaN,5111700000,UA51100410010044384,М,UA51000000000030770,Одеська область,UA51100000000095786,NaN,-1
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,м.Гаага,NaN,NaN,NaN,OC18030000000000000,NaN,OC00000000000000000,Інші країни,OC18000000000000000,NaN,-1
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,м.Воррен,NaN,NaN,NaN,OC26010000000000000,NaN,OC00000000000000000,Інші країни,OC26000000000000000,NaN,-1
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,м.Домброва-Гурнича,NaN,NaN,NaN,OC21050000000000000,NaN,OC00000000000000000,Інші країни,OC21000000000000000,NaN,-1


In [163]:
locations_2025.category.unique()

array(['С', 'М', 'В', 'Т', 'Х', nan], dtype=object)

In [164]:
all(el.startswith('OC') for el in locations_2025[locations_2025.KOATUU.isna()].KATOTTG.unique())

True

In [165]:
locations_2025 = locations_2025.drop(columns = ['third_level','third_level_code'], axis=1)
locations_2025

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG
0,Львівська область,Яворівський район,с.Гусаків,NaN,Яворівський район,NaN,с.Гусаків,NaN,NaN,4622481601,UA46140090120020894,С,UA46000000000026241,Львівська область,UA46140000000036328
1,Львівська область,Яворівський район,с.Баличі,NaN,Яворівський район,NaN,с.Баличі,NaN,NaN,4622480401,UA46140090020037836,С,UA46000000000026241,Львівська область,UA46140000000036328
2,Львівська область,Яворівський район,с.Шегині,NaN,Яворівський район,NaN,с.Шегині,NaN,NaN,4622487901,UA46140090010034963,С,UA46000000000026241,Львівська область,UA46140000000036328
3,Львівська область,Яворівський район,с.Волиця,NaN,Яворівський район,NaN,с.Волиця,NaN,NaN,4622480901,UA46140090090088787,С,UA46000000000026241,Львівська область,UA46140000000036328
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,NaN,Надвірнянський район,NaN,с.Гвізд,NaN,NaN,2624081501,UA26120070030065363,С,UA26000000000069363,Івано-Франківська область,UA26120000000036058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,NaN,м.Південне,NaN,м.Південне,NaN,NaN,5111700000,UA51100410010044384,М,UA51000000000030770,Одеська область,UA51100000000095786
9755,Нідерланди,м.Гаага,м.Гаага,NaN,Нідерланди,NaN,м.Гаага,NaN,NaN,NaN,OC18030000000000000,NaN,OC00000000000000000,Інші країни,OC18000000000000000
9756,США,м.Воррен,м.Воррен,NaN,США,NaN,м.Воррен,NaN,NaN,NaN,OC26010000000000000,NaN,OC00000000000000000,Інші країни,OC26000000000000000
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,NaN,Польща,NaN,м.Домброва-Гурнича,NaN,NaN,NaN,OC21050000000000000,NaN,OC00000000000000000,Інші країни,OC21000000000000000


In [166]:
locations_2025['KOATUU'] = locations_2025.apply(lambda row: '00'+row['KATOTTG'][2:10] if isinstance(row['KOATUU'], float) else row['KOATUU'], axis=1)
locations_2025['region_KOATUU'] = locations_2025['region_KOATUU'].fillna((locations_2025['KOATUU'].str[:2]+'00000000'))
locations_2025['area_KOATUU'] = locations_2025['area_KOATUU'].fillna((locations_2025['KOATUU'].str[:5]+'00000'))
locations_2025['category'] = locations_2025['category'].fillna('abroad')
locations_2025

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG
0,Львівська область,Яворівський район,с.Гусаків,4600000000,Яворівський район,4622400000,с.Гусаків,NaN,NaN,4622481601,UA46140090120020894,С,UA46000000000026241,Львівська область,UA46140000000036328
1,Львівська область,Яворівський район,с.Баличі,4600000000,Яворівський район,4622400000,с.Баличі,NaN,NaN,4622480401,UA46140090020037836,С,UA46000000000026241,Львівська область,UA46140000000036328
2,Львівська область,Яворівський район,с.Шегині,4600000000,Яворівський район,4622400000,с.Шегині,NaN,NaN,4622487901,UA46140090010034963,С,UA46000000000026241,Львівська область,UA46140000000036328
3,Львівська область,Яворівський район,с.Волиця,4600000000,Яворівський район,4622400000,с.Волиця,NaN,NaN,4622480901,UA46140090090088787,С,UA46000000000026241,Львівська область,UA46140000000036328
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,2600000000,Надвірнянський район,2624000000,с.Гвізд,NaN,NaN,2624081501,UA26120070030065363,С,UA26000000000069363,Івано-Франківська область,UA26120000000036058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,5100000000,м.Південне,5111700000,м.Південне,NaN,NaN,5111700000,UA51100410010044384,М,UA51000000000030770,Одеська область,UA51100000000095786
9755,Нідерланди,м.Гаага,м.Гаага,0000000000,Нідерланди,0018000000,м.Гаага,NaN,NaN,0018030000,OC18030000000000000,abroad,OC00000000000000000,Інші країни,OC18000000000000000
9756,США,м.Воррен,м.Воррен,0000000000,США,0026000000,м.Воррен,NaN,NaN,0026010000,OC26010000000000000,abroad,OC00000000000000000,Інші країни,OC26000000000000000
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,0000000000,Польща,0021000000,м.Домброва-Гурнича,NaN,NaN,0021050000,OC21050000000000000,abroad,OC00000000000000000,Інші країни,OC21000000000000000


In [167]:
locations_KATOTTG['region_KATOTTG'] =np.nan
locations_KATOTTG['area_KATOTTG'] =np.nan
locations_KATOTTG

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,area_KATOTTG
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230,с.Терпіння,23230,85101,2323085101,UA23080270010078454,С,NaN,NaN
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227,м.Красилів,68227,10100,6822710100,UA68040210010032567,М,NaN,NaN
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238,с.Дмитрівка,12238,81501,1223881501,UA12140170040016918,С,NaN,NaN
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101,м.Чернівці,73101,00000,7310100000,UA73060610010033137,М,NaN,NaN
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223,с.Кумарі,48223,83001,4822383001,UA48080050190079797,С,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22692,Житомирська область,Любарський район,с.Великий Браталів,18,Любарський район,231,с.Великий Браталів,18231,55106,1823155106,UA18040290070047586,С,NaN,NaN
22693,Хмельницька область,Ізяславський район,с.Влашанівка,68,Ізяславський район,221,с.Влашанівка,68221,86602,6822186602,UA68060250040099385,С,NaN,NaN
22694,Львівська область,Яворівський район,с.Коханівка,46,Яворівський район,258,с.Коханівка,46258,87803,4625887803,UA46140110360079694,С,NaN,NaN
22695,Житомирська область,Ємільчинський район,с.Бастова Рудня,18,Ємільчинський район,217,с.Бастова Рудня,18217,80403,1821780403,UA18080030040080438,С,NaN,NaN


In [168]:
locations_KATOTTG['area_KATOTTG'] = locations_KATOTTG['area_KATOTTG'].fillna(locations_KATOTTG['KATOTTG'].map(code2025[code2025.additional_level.notna()].set_index('additional_level')['second_level']))


In [169]:
locations_KATOTTG['area_KATOTTG'] = locations_KATOTTG['area_KATOTTG'].fillna(locations_KATOTTG['KATOTTG'].map(code2025[(code2025.fourth_level.notna())&(code2025.additional_level.isna())].set_index('fourth_level')['second_level']))
locations_KATOTTG

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,area_KATOTTG
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230,с.Терпіння,23230,85101,2323085101,UA23080270010078454,С,NaN,UA23080000000090746
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227,м.Красилів,68227,10100,6822710100,UA68040210010032567,М,NaN,UA68040000000086061
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238,с.Дмитрівка,12238,81501,1223881501,UA12140170040016918,С,NaN,UA12140000000011720
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101,м.Чернівці,73101,00000,7310100000,UA73060610010033137,М,NaN,UA73060000000061247
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223,с.Кумарі,48223,83001,4822383001,UA48080050190079797,С,NaN,UA48080000000082320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22692,Житомирська область,Любарський район,с.Великий Браталів,18,Любарський район,231,с.Великий Браталів,18231,55106,1823155106,UA18040290070047586,С,NaN,UA18040000000058965
22693,Хмельницька область,Ізяславський район,с.Влашанівка,68,Ізяславський район,221,с.Влашанівка,68221,86602,6822186602,UA68060250040099385,С,NaN,UA68060000000040570
22694,Львівська область,Яворівський район,с.Коханівка,46,Яворівський район,258,с.Коханівка,46258,87803,4625887803,UA46140110360079694,С,NaN,UA46140000000036328
22695,Житомирська область,Ємільчинський район,с.Бастова Рудня,18,Ємільчинський район,217,с.Бастова Рудня,18217,80403,1821780403,UA18080030040080438,С,NaN,UA18080000000016657


In [170]:
locations_KATOTTG['region_KATOTTG'] = locations_KATOTTG['region_KATOTTG'].fillna(locations_KATOTTG['KATOTTG'].map(code2025[code2025.additional_level.notna()].set_index('additional_level')['first_level']))


In [171]:
locations_KATOTTG['region_KATOTTG'] = locations_KATOTTG['region_KATOTTG'].fillna(locations_KATOTTG['KATOTTG'].map(code2025[(code2025.fourth_level.notna())&(code2025.additional_level.isna())].set_index('fourth_level')['first_level']))
locations_KATOTTG

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,area_KATOTTG
0,Запорізька область,Мелітопольський район,с.Терпіння,23,Мелітопольський район,230,с.Терпіння,23230,85101,2323085101,UA23080270010078454,С,UA23000000000064947,UA23080000000090746
1,Хмельницька область,Красилівський район,м.Красилів,68,Красилівський район,227,м.Красилів,68227,10100,6822710100,UA68040210010032567,М,UA68000000000099709,UA68040000000086061
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,12,Петропавлівський район,238,с.Дмитрівка,12238,81501,1223881501,UA12140170040016918,С,UA12000000000090473,UA12140000000011720
3,Чернівецька область,м.Чернівці,Шевченківський район міста,73,м.Чернівці,101,м.Чернівці,73101,00000,7310100000,UA73060610010033137,М,UA73000000000044923,UA73060000000061247
4,Миколаївська область,Врадіївський район,с.Кумарі,48,Врадіївський район,223,с.Кумарі,48223,83001,4822383001,UA48080050190079797,С,UA48000000000039575,UA48080000000082320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22692,Житомирська область,Любарський район,с.Великий Браталів,18,Любарський район,231,с.Великий Браталів,18231,55106,1823155106,UA18040290070047586,С,UA18000000000041385,UA18040000000058965
22693,Хмельницька область,Ізяславський район,с.Влашанівка,68,Ізяславський район,221,с.Влашанівка,68221,86602,6822186602,UA68060250040099385,С,UA68000000000099709,UA68060000000040570
22694,Львівська область,Яворівський район,с.Коханівка,46,Яворівський район,258,с.Коханівка,46258,87803,4625887803,UA46140110360079694,С,UA46000000000026241,UA46140000000036328
22695,Житомирська область,Ємільчинський район,с.Бастова Рудня,18,Ємільчинський район,217,с.Бастова Рудня,18217,80403,1821780403,UA18080030040080438,С,UA18000000000041385,UA18080000000016657


In [172]:
locations_KATOTTG['region_KOATUU'] = locations_KATOTTG.apply(lambda row: row['region_KOATUU']+'00000000' , axis=1)
locations_KATOTTG

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,area_KATOTTG
0,Запорізька область,Мелітопольський район,с.Терпіння,2300000000,Мелітопольський район,230,с.Терпіння,23230,85101,2323085101,UA23080270010078454,С,UA23000000000064947,UA23080000000090746
1,Хмельницька область,Красилівський район,м.Красилів,6800000000,Красилівський район,227,м.Красилів,68227,10100,6822710100,UA68040210010032567,М,UA68000000000099709,UA68040000000086061
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,1200000000,Петропавлівський район,238,с.Дмитрівка,12238,81501,1223881501,UA12140170040016918,С,UA12000000000090473,UA12140000000011720
3,Чернівецька область,м.Чернівці,Шевченківський район міста,7300000000,м.Чернівці,101,м.Чернівці,73101,00000,7310100000,UA73060610010033137,М,UA73000000000044923,UA73060000000061247
4,Миколаївська область,Врадіївський район,с.Кумарі,4800000000,Врадіївський район,223,с.Кумарі,48223,83001,4822383001,UA48080050190079797,С,UA48000000000039575,UA48080000000082320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22692,Житомирська область,Любарський район,с.Великий Браталів,1800000000,Любарський район,231,с.Великий Браталів,18231,55106,1823155106,UA18040290070047586,С,UA18000000000041385,UA18040000000058965
22693,Хмельницька область,Ізяславський район,с.Влашанівка,6800000000,Ізяславський район,221,с.Влашанівка,68221,86602,6822186602,UA68060250040099385,С,UA68000000000099709,UA68060000000040570
22694,Львівська область,Яворівський район,с.Коханівка,4600000000,Яворівський район,258,с.Коханівка,46258,87803,4625887803,UA46140110360079694,С,UA46000000000026241,UA46140000000036328
22695,Житомирська область,Ємільчинський район,с.Бастова Рудня,1800000000,Ємільчинський район,217,с.Бастова Рудня,18217,80403,1821780403,UA18080030040080438,С,UA18000000000041385,UA18080000000016657


In [173]:
locations_KATOTTG['area_KOATUU'] = locations_KATOTTG.apply(lambda row: row['code']+'00000' , axis=1)
locations_KATOTTG

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,area_KATOTTG
0,Запорізька область,Мелітопольський район,с.Терпіння,2300000000,Мелітопольський район,2323000000,с.Терпіння,23230,85101,2323085101,UA23080270010078454,С,UA23000000000064947,UA23080000000090746
1,Хмельницька область,Красилівський район,м.Красилів,6800000000,Красилівський район,6822700000,м.Красилів,68227,10100,6822710100,UA68040210010032567,М,UA68000000000099709,UA68040000000086061
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,1200000000,Петропавлівський район,1223800000,с.Дмитрівка,12238,81501,1223881501,UA12140170040016918,С,UA12000000000090473,UA12140000000011720
3,Чернівецька область,м.Чернівці,Шевченківський район міста,7300000000,м.Чернівці,7310100000,м.Чернівці,73101,00000,7310100000,UA73060610010033137,М,UA73000000000044923,UA73060000000061247
4,Миколаївська область,Врадіївський район,с.Кумарі,4800000000,Врадіївський район,4822300000,с.Кумарі,48223,83001,4822383001,UA48080050190079797,С,UA48000000000039575,UA48080000000082320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22692,Житомирська область,Любарський район,с.Великий Браталів,1800000000,Любарський район,1823100000,с.Великий Браталів,18231,55106,1823155106,UA18040290070047586,С,UA18000000000041385,UA18040000000058965
22693,Хмельницька область,Ізяславський район,с.Влашанівка,6800000000,Ізяславський район,6822100000,с.Влашанівка,68221,86602,6822186602,UA68060250040099385,С,UA68000000000099709,UA68060000000040570
22694,Львівська область,Яворівський район,с.Коханівка,4600000000,Яворівський район,4625800000,с.Коханівка,46258,87803,4625887803,UA46140110360079694,С,UA46000000000026241,UA46140000000036328
22695,Житомирська область,Ємільчинський район,с.Бастова Рудня,1800000000,Ємільчинський район,1821700000,с.Бастова Рудня,18217,80403,1821780403,UA18080030040080438,С,UA18000000000041385,UA18080000000016657


In [174]:
locations_2025

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,code,local_KOATUU,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG
0,Львівська область,Яворівський район,с.Гусаків,4600000000,Яворівський район,4622400000,с.Гусаків,NaN,NaN,4622481601,UA46140090120020894,С,UA46000000000026241,Львівська область,UA46140000000036328
1,Львівська область,Яворівський район,с.Баличі,4600000000,Яворівський район,4622400000,с.Баличі,NaN,NaN,4622480401,UA46140090020037836,С,UA46000000000026241,Львівська область,UA46140000000036328
2,Львівська область,Яворівський район,с.Шегині,4600000000,Яворівський район,4622400000,с.Шегині,NaN,NaN,4622487901,UA46140090010034963,С,UA46000000000026241,Львівська область,UA46140000000036328
3,Львівська область,Яворівський район,с.Волиця,4600000000,Яворівський район,4622400000,с.Волиця,NaN,NaN,4622480901,UA46140090090088787,С,UA46000000000026241,Львівська область,UA46140000000036328
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,2600000000,Надвірнянський район,2624000000,с.Гвізд,NaN,NaN,2624081501,UA26120070030065363,С,UA26000000000069363,Івано-Франківська область,UA26120000000036058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,5100000000,м.Південне,5111700000,м.Південне,NaN,NaN,5111700000,UA51100410010044384,М,UA51000000000030770,Одеська область,UA51100000000095786
9755,Нідерланди,м.Гаага,м.Гаага,0000000000,Нідерланди,0018000000,м.Гаага,NaN,NaN,0018030000,OC18030000000000000,abroad,OC00000000000000000,Інші країни,OC18000000000000000
9756,США,м.Воррен,м.Воррен,0000000000,США,0026000000,м.Воррен,NaN,NaN,0026010000,OC26010000000000000,abroad,OC00000000000000000,Інші країни,OC26000000000000000
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,0000000000,Польща,0021000000,м.Домброва-Гурнича,NaN,NaN,0021050000,OC21050000000000000,abroad,OC00000000000000000,Інші країни,OC21000000000000000


In [175]:
locations_KATOTTG.columns

Index(['regname', 'areaname', 'tername', 'region_KOATUU', 'areaname_new',
       'area_KOATUU', 'tername_new', 'code', 'local_KOATUU', 'KOATUU',
       'KATOTTG', 'category', 'region_KATOTTG', 'area_KATOTTG'],
      dtype='object')

In [176]:
locations_2025 = locations_2025.drop(columns = ['code', 'local_KOATUU'], axis=1)
locations_KATOTTG = locations_KATOTTG.drop(columns = ['code', 'local_KOATUU'], axis=1)
locations_2025

,regname,areaname,tername,region_KOATUU,areaname_new,area_KOATUU,tername_new,KOATUU,KATOTTG,category,region_KATOTTG,region_new,area_KATOTTG
0,Львівська область,Яворівський район,с.Гусаків,4600000000,Яворівський район,4622400000,с.Гусаків,4622481601,UA46140090120020894,С,UA46000000000026241,Львівська область,UA46140000000036328
1,Львівська область,Яворівський район,с.Баличі,4600000000,Яворівський район,4622400000,с.Баличі,4622480401,UA46140090020037836,С,UA46000000000026241,Львівська область,UA46140000000036328
2,Львівська область,Яворівський район,с.Шегині,4600000000,Яворівський район,4622400000,с.Шегині,4622487901,UA46140090010034963,С,UA46000000000026241,Львівська область,UA46140000000036328
3,Львівська область,Яворівський район,с.Волиця,4600000000,Яворівський район,4622400000,с.Волиця,4622480901,UA46140090090088787,С,UA46000000000026241,Львівська область,UA46140000000036328
4,Івано-Франківська область,Надвірнянський район,с.Гвізд,2600000000,Надвірнянський район,2624000000,с.Гвізд,2624081501,UA26120070030065363,С,UA26000000000069363,Івано-Франківська область,UA26120000000036058
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,Одеська область,м.Південне,м.Південне,5100000000,м.Південне,5111700000,м.Південне,5111700000,UA51100410010044384,М,UA51000000000030770,Одеська область,UA51100000000095786
9755,Нідерланди,м.Гаага,м.Гаага,0000000000,Нідерланди,0018000000,м.Гаага,0018030000,OC18030000000000000,abroad,OC00000000000000000,Інші країни,OC18000000000000000
9756,США,м.Воррен,м.Воррен,0000000000,США,0026000000,м.Воррен,0026010000,OC26010000000000000,abroad,OC00000000000000000,Інші країни,OC26000000000000000
9757,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,0000000000,Польща,0021000000,м.Домброва-Гурнича,0021050000,OC21050000000000000,abroad,OC00000000000000000,Інші країни,OC21000000000000000


In [177]:
locations_KATOTTG.columns

Index(['regname', 'areaname', 'tername', 'region_KOATUU', 'areaname_new',
       'area_KOATUU', 'tername_new', 'KOATUU', 'KATOTTG', 'category',
       'region_KATOTTG', 'area_KATOTTG'],
      dtype='object')

In [181]:
merged_df = pd.merge(locations_KATOTTG[['regname', 'areaname', 'tername', 'KOATUU', 'KATOTTG', 'category']], locations_2025[['regname', 'areaname', 'tername', 'KOATUU', 'KATOTTG', 'category']], on=['regname', 'areaname', 'tername', 'category'], suffixes=('_2021', '_2025'), how='inner')
merged_df

,regname,areaname,tername,KOATUU_2021,KATOTTG_2021,category,KOATUU_2025,KATOTTG_2025
0,Запорізька область,Мелітопольський район,с.Терпіння,2323085101,UA23080270010078454,С,2323085101,UA23080270010078454
1,Одеська область,м.Одеса,Суворовський район міста,5110137600,UA51100270010413116,В,5110137600,UA51100270010413116
2,Львівська область,Яворівський район,смт Івано-Франкове,4625855300,UA46140010010031231,Т,4625855300,UA46140010010031231
3,Одеська область,м.Одеса,Малиновський район міста,5110137300,UA51100270010275193,В,5110137300,UA51100270010275193
4,Львівська область,Дрогобицький район,с.Рихтичі,4621286301,UA46020030280042854,С,4621286301,UA46020030280042854
...,...,...,...,...,...,...,...,...
2310,Сумська область,Роменський район,с.Авраменкове,5924188503,UA59060110020098714,С,5924188503,UA59060110020098714
2311,Київська область,Білоцерківський район,с.Сорокотяги,3220486201,UA32020090130055138,С,3220486201,UA32020090130055138
2312,Рівненська область,Дубенський район,с.Нагоряни,5621688207,UA56040310140028483,С,5621688207,UA56040310140028483
2313,Житомирська область,Новоград-Волинський район,с.Кануни,1824085402,UA18080190040044862,С,1824085402,UA18080190040044862


In [182]:
df1_cols = [col for col in merged_df.columns if col.endswith('_2021')][3:]
df2_cols = [col for col in merged_df.columns if col.endswith('_2025')][3:]


different_rows = merged_df[(merged_df[df1_cols].values != merged_df[df2_cols].values).any(axis=1)]

if not different_rows.empty:
    print("There are rows with equal values in the first 3 columns and different values in other columns.")
    display(different_rows)
else:
    print("No rows found that meet the condition.")

No rows found that meet the condition.


In [209]:
concatenated_df = pd.concat([locations_KATOTTG[['regname', 'areaname', 'tername', 'KOATUU', 'KATOTTG', 'category']], locations_2025[['regname', 'areaname', 'tername', 'KOATUU', 'KATOTTG','category']]], ignore_index=True)
concatenated_df.drop_duplicates(inplace=True)
concatenated_df.reset_index(drop=True, inplace=True)


concatenated_df
# TODO: urban-rural status
# TODO: translation regname ukr_regname, eng_regname,...
# TODO: update dependencies (new location of notebook)

,regname,areaname,tername,KOATUU,KATOTTG,category
0,Запорізька область,Мелітопольський район,с.Терпіння,2323085101,UA23080270010078454,С
1,Хмельницька область,Красилівський район,м.Красилів,6822710100,UA68040210010032567,М
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,1223881501,UA12140170040016918,С
3,Чернівецька область,м.Чернівці,Шевченківський район міста,7310100000,UA73060610010033137,М
4,Миколаївська область,Врадіївський район,с.Кумарі,4822383001,UA48080050190079797,С
...,...,...,...,...,...,...
30135,Одеська область,м.Південне,м.Південне,5111700000,UA51100410010044384,М
30136,Нідерланди,м.Гаага,м.Гаага,0018030000,OC18030000000000000,abroad
30137,США,м.Воррен,м.Воррен,0026010000,OC26010000000000000,abroad
30138,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,0021050000,OC21050000000000000,abroad


In [210]:
concatenated_df.category.unique()

array(['С', 'М', 'В', 'Т', 'Х', 'abroad'], dtype=object)

In [211]:
dict_category = {'С':'village', 'Х':'settlement', 'Т':'urban village', 'М':'town', 'В':'city'}

for abbr, name in dict_category.items():
    concatenated_df.loc[concatenated_df.category == abbr, 'category'] = name

In [212]:
concatenated_df.category.unique()

array(['village', 'town', 'city', 'urban village', 'settlement', 'abroad'],
      dtype=object)

In [213]:
concatenated_df[concatenated_df.KATOTTG.str.len() !=len('OC13020000000000000')]

,regname,areaname,tername,KOATUU,KATOTTG,category


In [214]:
concatenated_df

,regname,areaname,tername,KOATUU,KATOTTG,category
0,Запорізька область,Мелітопольський район,с.Терпіння,2323085101,UA23080270010078454,village
1,Хмельницька область,Красилівський район,м.Красилів,6822710100,UA68040210010032567,town
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,1223881501,UA12140170040016918,village
3,Чернівецька область,м.Чернівці,Шевченківський район міста,7310100000,UA73060610010033137,town
4,Миколаївська область,Врадіївський район,с.Кумарі,4822383001,UA48080050190079797,village
...,...,...,...,...,...,...
30135,Одеська область,м.Південне,м.Південне,5111700000,UA51100410010044384,town
30136,Нідерланди,м.Гаага,м.Гаага,0018030000,OC18030000000000000,abroad
30137,США,м.Воррен,м.Воррен,0026010000,OC26010000000000000,abroad
30138,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,0021050000,OC21050000000000000,abroad


In [215]:
concatenated_df['region_code'] = concatenated_df['KATOTTG'].str[:4]
sorted(concatenated_df.region_code.unique())

['OC01',
 'OC02',
 'OC03',
 'OC04',
 'OC05',
 'OC06',
 'OC07',
 'OC08',
 'OC09',
 'OC10',
 'OC11',
 'OC12',
 'OC13',
 'OC14',
 'OC15',
 'OC16',
 'OC17',
 'OC18',
 'OC19',
 'OC20',
 'OC21',
 'OC22',
 'OC23',
 'OC24',
 'OC25',
 'OC26',
 'OC27',
 'OC28',
 'OC29',
 'OC30',
 'OC31',
 'OC32',
 'OC33',
 'OC34',
 'UA05',
 'UA07',
 'UA12',
 'UA14',
 'UA18',
 'UA21',
 'UA23',
 'UA26',
 'UA32',
 'UA35',
 'UA44',
 'UA46',
 'UA48',
 'UA51',
 'UA53',
 'UA56',
 'UA59',
 'UA61',
 'UA63',
 'UA65',
 'UA68',
 'UA71',
 'UA73',
 'UA74',
 'UA80']

In [216]:
concatenated_df['region_name'] = concatenated_df['region_code'].apply(lambda x: renam.dct_code_regions[x])
concatenated_df

,regname,areaname,tername,KOATUU,KATOTTG,category,region_code,region_name
0,Запорізька область,Мелітопольський район,с.Терпіння,2323085101,UA23080270010078454,village,UA23,Zaporizka
1,Хмельницька область,Красилівський район,м.Красилів,6822710100,UA68040210010032567,town,UA68,Khmelnytska
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,1223881501,UA12140170040016918,village,UA12,Dnipropetrovska
3,Чернівецька область,м.Чернівці,Шевченківський район міста,7310100000,UA73060610010033137,town,UA73,Chernivetska
4,Миколаївська область,Врадіївський район,с.Кумарі,4822383001,UA48080050190079797,village,UA48,Mykolaivska
...,...,...,...,...,...,...,...,...
30135,Одеська область,м.Південне,м.Південне,5111700000,UA51100410010044384,town,UA51,Odeska
30136,Нідерланди,м.Гаага,м.Гаага,0018030000,OC18030000000000000,abroad,OC18,Netherlands
30137,США,м.Воррен,м.Воррен,0026010000,OC26010000000000000,abroad,OC26,United States of America
30138,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,0021050000,OC21050000000000000,abroad,OC21,Poland


In [217]:
concatenated_df = concatenated_df.drop(columns = ['region_code'], axis=1)
concatenated_df

,regname,areaname,tername,KOATUU,KATOTTG,category,region_name
0,Запорізька область,Мелітопольський район,с.Терпіння,2323085101,UA23080270010078454,village,Zaporizka
1,Хмельницька область,Красилівський район,м.Красилів,6822710100,UA68040210010032567,town,Khmelnytska
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,1223881501,UA12140170040016918,village,Dnipropetrovska
3,Чернівецька область,м.Чернівці,Шевченківський район міста,7310100000,UA73060610010033137,town,Chernivetska
4,Миколаївська область,Врадіївський район,с.Кумарі,4822383001,UA48080050190079797,village,Mykolaivska
...,...,...,...,...,...,...,...
30135,Одеська область,м.Південне,м.Південне,5111700000,UA51100410010044384,town,Odeska
30136,Нідерланди,м.Гаага,м.Гаага,0018030000,OC18030000000000000,abroad,Netherlands
30137,США,м.Воррен,м.Воррен,0026010000,OC26010000000000000,abroad,United States of America
30138,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,0021050000,OC21050000000000000,abroad,Poland


In [220]:
dup_katottg = concatenated_df["KATOTTG"].value_counts()
dup_katottg = dup_katottg[dup_katottg > 1]
dup_katottg

KATOTTG
UA73060610010033137    9
UA23040050010097416    9
UA05020030010063857    9
UA35040210010145346    7
UA35040210010286392    7
                      ..
UA65080050030084440    2
UA14140030160031433    2
UA23040130020042658    2
UA56040090020088413    2
UA63120030020029600    2
Name: count, Length: 9300, dtype: int64

In [222]:
concatenated_df[concatenated_df["KATOTTG"] =='UA23040050010097416']

,regname,areaname,tername,KOATUU,KATOTTG,category,region_name
872,Запорізька область,Великобілозерський район,с.Велика Білозерка(Частина 4 Села) (Червона сі...,2321180101,UA23040050010097416,village,Zaporizka
5484,Запорізька область,Великобілозерський район,с.Велика Білозерка(Частина 1 Села),2321180101,UA23040050010097416,village,Zaporizka
10001,Запорізька область,Великобілозерський район,с.Велика Білозерка(Частина 2 Села) (Новопетрів...,2321180101,UA23040050010097416,village,Zaporizka
15565,Запорізька область,Великобілозерський район,с.Велика Білозерка(Частина 3 Села) (Трудова сі...,2321180101,UA23040050010097416,village,Zaporizka
17249,Запорізька область,Великобілозерський район,с.Велика Білозерка(Частина 4 Села),2321180101,UA23040050010097416,village,Zaporizka
17681,Запорізька область,Великобілозерський район,с.Велика Білозерка(Частина 3 Села),2321180101,UA23040050010097416,village,Zaporizka
18940,Запорізька область,Великобілозерський район,с.Велика Білозерка(Частина 2 Села),2321180101,UA23040050010097416,village,Zaporizka
21796,Запорізька область,Великобілозерський район,с.Велика Білозерка,2321180101,UA23040050010097416,village,Zaporizka
23196,Запорізька область,Василівський район,с.Велика Білозерка,2321180101,UA23040050010097416,village,Zaporizka


In [194]:
concatenated_df.region_name.unique()

array(['Zaporizka', 'Khmelnytska', 'Dnipropetrovska', 'Chernivetska',
       'Mykolaivska', 'Donetska', 'Ternopilska', 'Kirovohradska',
       'Odeska', 'Lvivska', 'Ivano-Frankivska', 'Kharkivska', 'Cherkaska',
       'Volynska', 'Poltavska', 'Rivnenska', 'Vinnytska', 'Chernihivska',
       'Kyivska', 'Khersonska', 'Zhytomyrska', 'Zakarpatska', 'Kyiv',
       'Sumska', 'Luhanska', 'Poland', 'France', 'Germany', 'Estonia',
       'Norway', 'Slovakia', 'Austia', 'United Kingdom', 'Ireland',
       'Bulgaria', 'Moldova', 'Portugal', 'Azerbaijan', 'Hungary',
       'Italy', 'Turkey', 'Czechia', 'Belgium', 'Spain', 'Canada',
       'Croatia', 'Latvia', 'Romania', 'Lithuania', 'Georgia',
       'United States of America', 'Finland', 'Sweden', 'Luxembourg',
       'Netherlands', 'Denmark', 'Switzerland', 'Slovenia', 'Greece'],
      dtype=object)

In [223]:
concatenated_df = concatenated_df.drop_duplicates(subset=['regname', 'areaname', 'tername'], keep='first')

In [224]:
concatenated_df.to_csv('./matching_data/locations.csv', index=False, encoding='utf-8')

In [225]:
concatenated_df_code = concatenated_df.drop(columns = ['regname', 'areaname','tername'], axis=1).drop_duplicates().reset_index(drop=True)
concatenated_df_code

,KOATUU,KATOTTG,category,region_name
0,2323085101,UA23080270010078454,village,Zaporizka
1,6822710100,UA68040210010032567,town,Khmelnytska
2,1223881501,UA12140170040016918,village,Dnipropetrovska
3,7310100000,UA73060610010033137,town,Chernivetska
4,4822383001,UA48080050190079797,village,Mykolaivska
...,...,...,...,...
18506,0003010000,OC03010000000000000,abroad,Italy
18507,0034010000,OC34010000000000000,abroad,Sweden
18508,0024020000,OC24020000000000000,abroad,Slovakia
18509,0018040000,OC18040000000000000,abroad,Netherlands


In [227]:
concatenated_df_code.to_csv('./final_tables/locations.csv', index=False, encoding='utf-8')